# Prithvi Reference-Data Assessment

**Project:** Prithvi-Based Landscape Change Attribution Service in Support of National Park Service and National Forest Monitoring Needs (NASA-funded, PI Robert Kennedy, Oregon State University).

This notebook is the visual/reproducible companion to `docs/data_inventory.md`, `docs/DATA_STATUS.md`, and the QA reports under `outputs/qa/`. It serves two purposes:

1. **Interactive visual QA** while the reference-data assessment methodology is being developed.
2. **The basis for the eventual reference-data assessment report**, exportable to HTML/PDF once the assessment is further along.

**This notebook reads already-produced outputs — it does not repeat processing logic.** Geometry repair and standardization happen in `src/process_nccn.py`; this notebook only loads the resulting `data/processed/` and `outputs/qa/` products and visualizes them. No source data is modified here. Reusable plotting/mapping code lives in `src/report_viz.py`, kept out of the notebook so this stays a short, readable narrative rather than an implementation file.

## Report structure

- **NCCN** — complete as of this version (this notebook)
- GLKN — *to be added*
- ADS Region 6 — *to be added*
- ADS Region 10 — *to be added*
- Cross-source comparison — *to be added*
- Conclusions / recommendations — *to be added*

## A note on interactive vs. static maps

Interactive maps (via [folium](https://python-visualization.github.io/folium/)) let you zoom, pan, and toggle layers on/off — but they do not render in a static PDF export of this notebook. Every interactive map below has a **static matplotlib equivalent** alongside it, suitable for the eventual exported report. Use the interactive maps while working in Jupyter; the static maps are what will actually appear in any HTML/PDF export.


In [ ]:
# Setup — all paths are relative to the repository root, never hardcoded.
# Works whether Jupyter was launched from the repo root or from notebooks/.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src").exists(), (
    f"Could not locate repo root (looked at {REPO_ROOT}); "
    "launch Jupyter from the repo root or from notebooks/."
)

sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

import report_viz as viz

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
QA_DIR = REPO_ROOT / "outputs" / "qa"

PARKS = ["MORA", "NOCA", "OLYM", "LEWI"]
PARK_NAMES = {
    "MORA": "Mount Rainier National Park",
    "NOCA": "North Cascades National Park",
    "OLYM": "Olympic National Park",
    "LEWI": "Lewis and Clark National Historical Park",
}

print(f"Repo root: {REPO_ROOT}")


---
# NCCN

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**What we're evaluating:** whether the four NCCN (North Coast and Cascades Network) attributed landscape-change datasets — Mount Rainier (MORA), North Cascades (NOCA), Olympic (OLYM), and Lewis & Clark (LEWI) — can be turned into a reliable, geometry-clean reference dataset, and what geographic unit (park boundary, HUC watershed, or something else) actually explains the extent of the monitored/attributed data.

**Datasets and versions used** (see `docs/DATA_MANIFEST.md` for exact paths):

| Park | Source dataset (schema) | Notes |
|---|---|---|
| MORA | `MORA_1987_2017_V2_1_1_UTM` (current, V2.1.1) | |
| NOCA | `NOCA_1987_2017_V2_1_1_UTM` (current, V2.1.1) | legacy `V2B` version exists in `data/raw/` but is **not** used here |
| OLYM | `OLYM_1987_2017_V2_1_1_UTM` (current, V2.1.1) | legacy `V2B` version exists in `data/raw/` but is **not** used here |
| LEWI | `LEWI_1985_2011_Report_UTM` (its own, older schema vintage — "V2A") | no current-schema (V2.1.1) update exists for LEWI |

**Raw vs. processed:** raw source data (`data/raw/nccn/`) is preserved exactly as received and is never modified. All repair/standardization happens in `src/process_nccn.py`, writing to `data/processed/nccn/` and `data/processed/boundaries/`, which is what this notebook reads. Full processing decisions and QA are documented in `outputs/qa/nccn_processing_report.md`.


### Dataset / Methods (as provided by NCCN)

The following is **source-provided methodological context**, reproduced here rather than paraphrased, using NCCN's own terminology for the change categories. It explains how the disturbance patches were generated and classified — **it does not by itself define the outer monitoring/study-area boundary** (that question is investigated separately in Section 7; nothing about the boundary should be inferred from this description).

> - NCCN developed the landscape-change protocol as part of NPS Vital Signs Monitoring.
> - Initial implementations were NOCA (2012), MORA (2013), and OLYM (2014), using OSU/eMapR LandTrendr.
> - The supplied dataset was generated from LandTrendr for 1987–2017.
> - Disturbance pixels for each year were aggregated into patches using adjacency rules and a minimum mapping unit of 0.8 ha (2 acres).
> - Those candidate disturbance patches were subsequently human reviewed and labeled.
> - MORA and NOCA use eight change categories: Avalanche, Blowdown, Clearing, Defoliation, Development, Fire, Mass Movement, and Riparian Change.
> - OLYM additionally includes Ice Damage and Coastal Change.

**Important distinction:** these are **human-interpreted LandTrendr disturbance patches**, produced by an analyst reviewing and labeling algorithm-flagged candidate patches — **not a wall-to-wall land-cover map**. There is no claim, implied or otherwise, that every pixel outside a mapped patch was reviewed and confirmed unchanged; absence of a patch is not evidence of no disturbance (see also `docs/data_inventory.md` design principle: "unlabeled ≠ no change").

**Scope note:** this description covers MORA, NOCA, and OLYM (the three "initial implementations" above, all on the current V2.1.1 schema, LandTrendr 1987–2017). **LEWI is a separate dataset with its own, earlier implementation history and schema vintage** (see Section 1 table above) and is not covered by this description — its year range and class vocabulary are checked separately below and are expected to differ.

In [ ]:
# Confirm raw data is untouched and processed products exist
raw_mora = DATA_RAW / "nccn" / "NCCN_Landscape_Change_LPa01_1987-2017_V2_1_1_DISTRIBUTION" / "MORA_1987_2017_V2_1_1_UTM.shp"
std_path = DATA_PROCESSED / "nccn" / "nccn_standardized.parquet"
boundary_path = DATA_PROCESSED / "boundaries" / "nccn_park_boundaries.parquet"

print("Raw MORA shapefile present:", raw_mora.exists())
print("Standardized NCCN parquet present:", std_path.exists())
print("NCCN park boundaries parquet present:", boundary_path.exists())

nccn = gpd.read_parquet(std_path)
boundaries = gpd.read_parquet(boundary_path)
huc10 = gpd.read_file(DATA_RAW / "boundaries" / "nps" / "Prithvi_NCCN" / "NCCN_HUC10.shp").to_crs(nccn.crs)
huc12 = gpd.read_file(DATA_RAW / "boundaries" / "nps" / "Prithvi_NCCN" / "NCCN_HUC12.shp").to_crs(nccn.crs)

print(f"\nLoaded {len(nccn)} standardized NCCN reference polygons, CRS={nccn.crs.to_epsg()}")
print(f"Loaded {len(boundaries)} NPS park boundaries")
print(f"Loaded {len(huc10)} HUC10 candidate watersheds")
print(f"Loaded {len(huc12)} HUC12 candidate watersheds")


In [ ]:
# QA check: does the processed data actually match the described protocol
# (1987-2017, and the class vocabularies above)? Flagged, not silently
# reconciled, per the source-provided methodological context above.
PROTOCOL_YEAR_RANGE = (1987, 2017)
PROTOCOL_CLASSES = {
    "MORA": {"Avalanche", "Blowdown", "Clearing", "Defoliation", "Development", "Fire", "Mass Movement", "Riparian Change"},
    "NOCA": {"Avalanche", "Blowdown", "Clearing", "Defoliation", "Development", "Fire", "Mass Movement", "Riparian Change"},
    "OLYM": {"Avalanche", "Blowdown", "Clearing", "Defoliation", "Development", "Fire", "Mass Movement",
             "Riparian Change", "Ice Damage", "Coastal Change"},
}

print("Year range check against the described 1987-2017 protocol period:")
for park in PARKS:
    sub = nccn[nccn["park_code"] == park]
    oor = sub[(sub["year"] < PROTOCOL_YEAR_RANGE[0]) | (sub["year"] > PROTOCOL_YEAR_RANGE[1])]
    flag = "" if len(oor) == 0 else "  <-- FLAGGED: outside described range"
    print(f"  {park}: data spans {sub['year'].min()}-{sub['year'].max()}, "
          f"{len(oor)} of {len(sub)} rows outside 1987-2017{flag}")

print("\nchange_class check against the described category lists:")
for park in PARKS:
    sub = nccn[nccn["park_code"] == park]
    if park not in PROTOCOL_CLASSES:
        print(f"  {park}: not covered by this protocol description (separate dataset/vintage) -- not checked")
        continue
    found = set(sub["change_class"].unique())
    unexpected = sorted(found - PROTOCOL_CLASSES[park])
    absent = sorted(PROTOCOL_CLASSES[park] - found)
    print(f"  {park}: unexpected classes = {unexpected or 'none'}; "
          f"described classes absent from data = {absent or 'none'}")


**Result:** MORA, NOCA, and OLYM match the described protocol exactly — year range 1987–2017 with zero out-of-range rows, and zero unexpected or missing classes relative to the described category lists, for all three parks. **LEWI's data (1985–2011) falls partly outside the 1987–2017 range** — expected and not an error, since LEWI is explicitly a separate, earlier dataset not covered by this description; it is flagged here rather than silently excluded from the check.

### A.2 Basic dataset summary (all 4 parks combined)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(nccn),
    "attributed_area_ha": nccn.geometry.area.sum() / 1e4,
    "first_year": int(nccn["year"].min()),
    "last_year": int(nccn["year"].max()),
    "years_represented": int(nccn["year"].nunique()),
    "n_native_change_classes": int(nccn["change_class"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `change_class` | Standardized field = native `ChangeType` (or `Chnge_type` for LEWI) -- the primary disturbance-type label assigned by an NCCN analyst |
| `year` | Standardized field = `Detect_yr` / `AnalysisYr` -- year LandTrendr detected the spectral change / assigned for analysis |
| `Confidence` | Analyst's confidence in the `change_class` label: 1 (least) - 3 (most) |
| `Alt_type` | Secondary/alternative change-type label, recorded when `Confidence` is 1 or 2 |
| `In_Park` | Y/N -- whether the polygon centroid falls inside the strict NPS park boundary (most attributed area does not, see Section 5) |
| `Dist_year` / `Dist_name` | Fire-specific fields: actual disturbance year and event name, used to group multiple LandTrendr patches that represent one real fire event |
| `PatchID` / `Patch_name` | Unique per-polygon identifier |
| `DPL` | Data-lineage flag (e.g., "Updated" for records added/changed after initial certification) |


### A.4 Overall native attribution distribution (change_class, all parks combined)

In [ ]:
overall_class_counts = nccn["change_class"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
overall_class_counts.plot(kind="barh", ax=ax, color="#4575b4")
ax.set_xlabel("attributed polygons (all 4 parks combined)")
ax.set_title("NCCN native change_class distribution, all parks combined (unharmonized)")
ax.invert_yaxis()
fig.tight_layout()
overall_class_counts


### A.5 Overall temporal distribution (all parks combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "nccn_by_park_year.csv")
year_area = by_year.groupby("year")["area_m2"].sum() / 1e4

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("attributed area (ha)")
ax.set_xlabel("year")
ax.set_title(f"NCCN attributed area by year, all parks combined "
             f"({int(by_year.year.min())}-{int(by_year.year.max())})")
fig.tight_layout()


### A.6 Overall spatial distribution (full dataset)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, park in zip(axes, PARKS):
    sub = nccn[nccn["park_code"] == park]
    sub.plot(ax=ax, color="#4575b4", edgecolor="none", alpha=0.6)
    ax.set_title(f"{park} (n={len(sub):,})", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("NCCN attributed change polygons, per park (full dataset, no boundary overlay)")
fig.tight_layout()


---
## PART B --- Regional / Zonal Assessment

Having characterized NCCN as a whole (Part A), we now divide its landscape into **4 park-level analysis regions** -- each park's own attributed-data footprint (Sections 5-6 below investigate, and do not resolve, what a better-defined containing boundary might be) -- and examine how attributed change, area, and native attribution classes are distributed, and change through time, within each.

### B.1 Regional summary (one row per park)

In [ ]:
feature_counts = nccn.groupby("park_code").size().reindex(PARKS).rename("feature_count")
year_ranges = nccn.groupby("park_code")["year"].agg(["min", "max"]).reindex(PARKS)
inventory = pd.concat([feature_counts, year_ranges], axis=1)
inventory.columns = ["feature_count", "year_min", "year_max"]

# Fuller regional summary -- one row per park (the analysis region), from
# src/nccn_regional_summary.py (reads the already-standardized parquet, no
# new raw processing)
park_summary = pd.read_csv(QA_DIR / "nccn_by_park_summary.csv").set_index("park_code").reindex(PARKS)
park_summary = park_summary.assign(attributed_area_ha=lambda d: d["attributed_area_m2"] / 1e4)[
    ["record_count", "attributed_area_ha", "first_year", "last_year", "years_represented", "native_class_count"]
]
park_summary


In [ ]:
area_ha = park_summary["attributed_area_ha"].reindex(PARKS)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(PARKS, area_ha.values, color="#d73027")
ax.set_ylabel("attributed area (ha)")
ax.set_title("NCCN attributed area by park")
for i, park in enumerate(PARKS):
    ax.text(i, area_ha[park] + area_ha.max() * 0.02, f"{area_ha[park]:,.0f}", ha="center", fontsize=8)
fig.tight_layout()


### B.2 Region through time

In [ ]:
# Years represented -- histogram per park
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=False)
for ax, park in zip(axes, PARKS):
    sub = nccn[nccn["park_code"] == park]
    ax.hist(sub["year"], bins=range(int(sub["year"].min()), int(sub["year"].max()) + 2), color="#4575b4")
    ax.set_title(park)
    ax.set_xlabel("year")
fig.suptitle("NCCN reference polygons by year, per park")
fig.tight_layout()


In [ ]:
year_area_region = pd.read_csv(QA_DIR / "nccn_by_park_year.csv")
heat = year_area_region.pivot_table(index="park_code", columns="year", values="area_m2", aggfunc="sum").fillna(0) / 1e4
heat = heat.reindex(PARKS)

fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=9)
ax.set_xlabel("year")
ax.set_title("NCCN attributed area (ha) by park x year")
fig.colorbar(im, ax=ax, label="attributed area (ha)")
fig.tight_layout()


### B.3 Class composition by region

In [ ]:
# ChangeType classes per park (raw change_class values, preserved verbatim -- no cross-source harmonization)
class_table = pd.crosstab(nccn["change_class"], nccn["park_code"])[PARKS].fillna(0).astype(int)
class_table


In [ ]:
by_class = pd.read_csv(QA_DIR / "nccn_by_park_class.csv")
class_pivot = by_class.pivot(index="park_code", columns="change_class", values="area_m2").reindex(PARKS).fillna(0)
class_pivot_pct = class_pivot.div(class_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
class_pivot_pct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("% of park's attributed area")
ax.set_title("NCCN native change_class composition by park (unharmonized -- native per-park vocabulary)")
ax.legend(title="change_class", fontsize=7, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()


### B.4 Class composition through time within region

In [ ]:
stack = pd.read_csv(QA_DIR / "nccn_by_park_year_class.csv")

TOP_N = 8
top_classes = stack.groupby("change_class")["area_m2"].sum().sort_values(ascending=False).head(TOP_N).index.tolist()
stack["class_bucket"] = stack["change_class"].where(stack["change_class"].isin(top_classes), "Other")

palette = plt.get_cmap("tab10").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=False)
for ax, park in zip(axes, PARKS):
    sub = stack[stack["park_code"] == park]
    pivot = sub.groupby(["year", "class_bucket"])["area_m2"].sum().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot_pct.columns:
            continue
        vals = pivot_pct[cls].values
        ax.bar(pivot_pct.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(park, fontsize=9)
    ax.tick_params(labelsize=6)

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("NCCN: change_class composition (% of attributed area) by year, within each park (top 8 + Other)", y=1.05)
fig.tight_layout()


**Note:** Sections 3-6 below (Geometry QA, Interactive Maps, Park-Boundary Comparison, HUC Exploratory Test) are detailed processing QA and boundary-definition investigation that informed the Part A/B conclusion that each park's own attributed-data footprint is used as its analysis region (no agreed containing boundary is currently in hand for MORA/NOCA/LEWI -- see Section 7.2). Preserved here as useful exploratory material -- candidate for moving to an appendix in the final report.

## 3. Geometry QA

Full detail: `outputs/qa/nccn_processing_report.md`. Repair method: `GeoSeries.make_valid()` (GEOS's topology-preserving repair, not the older `buffer(0)` trick).

In [ ]:
before = pd.read_csv(QA_DIR / "nccn_geometry_validity_before_repair.csv").set_index("park_code").reindex(PARKS)
after = pd.read_csv(QA_DIR / "nccn_geometry_repair_qa.csv").set_index("park_code").reindex(PARKS)

validity_table = before[["feature_count", "valid_count", "invalid_count", "invalid_pct"]]
validity_table


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(PARKS, before.loc[PARKS, "invalid_pct"], color="#d73027")
ax.set_ylabel("% invalid geometries (before repair)")
ax.set_title("NCCN geometry invalidity by park, before repair")
for i, park in enumerate(PARKS):
    ax.text(i, before.loc[park, "invalid_pct"] + 1, f"{before.loc[park, 'invalid_pct']:.1f}%", ha="center")
fig.tight_layout()


In [ ]:
# Before/after area comparison -- demonstrates make_valid() produced negligible area change
area_compare = after[["total_area_m2_before", "total_area_m2_after", "area_diff_m2", "area_pct_diff",
                        "geometry_type_changed_count", "multipart_before_count", "multipart_after_count",
                        "empty_after_repair_count", "large_change_feature_count"]]
area_compare


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = range(len(PARKS))
width = 0.35
ax.bar([i - width/2 for i in x], after.loc[PARKS, "total_area_m2_before"] / 1e6, width, label="before repair", color="#4575b4")
ax.bar([i + width/2 for i in x], after.loc[PARKS, "total_area_m2_after"] / 1e6, width, label="after repair", color="#91bfdb")
ax.set_xticks(list(x)); ax.set_xticklabels(PARKS)
ax.set_ylabel("total area (km²)")
ax.set_title("Total reference-polygon area before vs. after geometry repair")
ax.legend()
fig.tight_layout()
print("Max |area % diff| across all four parks:", area_compare["area_pct_diff"].abs().max(), "%")


**Result: `make_valid()` produced negligible area change** — the maximum absolute area difference across all four parks, printed above, is effectively zero (floating-point noise, ~10⁻⁷%), despite invalidity rates as high as 80% (MORA) before repair. Zero features collapsed into a mixed-type `GeometryCollection`, zero went empty, and zero of 12,630 features exceeded a 5% individual-area-change threshold (`outputs/qa/nccn_geometry_repair_large_changes.csv` — header only, no rows). This is a **measured result**, not an assumption — see `outputs/qa/nccn_processing_report.md` for the full per-feature check.

## 4. Interactive / Visual Maps

For each park: NCCN reference polygons (red), the official NPS park boundary (black outline), and the HUC10 (blue) and HUC12 (dashed purple) watersheds currently being evaluated as possible study-area geographies. Interactive maps below are for exploratory use in Jupyter (zoom/pan, toggle layers via the control in the top-right); a static grid of the same four maps follows for report/PDF use.

In [ ]:
# MORA -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "MORA"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "MORA"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# NOCA -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "NOCA"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "NOCA"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# OLYM -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "OLYM"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "OLYM"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# LEWI -- interactive map (folium). Toggle layers with the control in the top-right.
ref_park = nccn[nccn["park_code"] == "LEWI"]
boundary_park = boundaries[boundaries["UNIT_CODE"] == "LEWI"]
huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]

m = viz.interactive_overlay_map([
    (huc10_park, viz.park_layer_style("huc10"), "HUC10 watersheds", True),
    (huc12_park, viz.park_layer_style("huc12"), "HUC12 watersheds", False),
    (boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (ref_park, viz.park_layer_style("reference"), "NCCN reference polygons", True),
])
m


In [ ]:
# Static equivalent of the four interactive maps above, for HTML/PDF export
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, park in zip(axes.flat, PARKS):
    ref_park = nccn[nccn["park_code"] == park]
    boundary_park = boundaries[boundaries["UNIT_CODE"] == park]
    huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    viz.static_overlay_map([
        (huc10_park, viz.park_layer_style_mpl("huc10"), "HUC10 watersheds"),
        (huc12_park, viz.park_layer_style_mpl("huc12"), "HUC12 watersheds"),
        (boundary_park, viz.park_layer_style_mpl("park_boundary"), "NPS park boundary"),
        (ref_park, viz.park_layer_style_mpl("reference"), "NCCN reference polygons"),
    ], title=f"{park} -- {PARK_NAMES[park]}", ax=ax)
fig.suptitle("NCCN reference polygons, NPS park boundary, and HUC10/HUC12 watersheds", y=1.01, fontsize=13)
fig.tight_layout()


## 5. Park-Boundary Comparison

Measured in `src/process_nccn.py` Step 5, full numbers in `outputs/qa/nccn_boundary_relationship_qa.csv`.

In [ ]:
bqa = pd.read_csv(QA_DIR / "nccn_boundary_relationship_qa.csv").set_index("park_code").reindex(PARKS)
bqa[["total_reference_polygons", "entirely_inside_count", "straddling_boundary_count",
     "entirely_outside_count", "pct_area_inside", "pct_area_outside"]]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(PARKS, bqa.loc[PARKS, "pct_area_inside"], label="% area inside park", color="#1a9850")
ax.bar(PARKS, bqa.loc[PARKS, "pct_area_outside"], bottom=bqa.loc[PARKS, "pct_area_inside"],
       label="% area outside park", color="#d73027")
ax.set_ylabel("% of total reference area")
ax.set_title("NCCN reference-polygon area: inside vs. outside the NPS park boundary")
ax.legend(loc="lower right")
for i, park in enumerate(PARKS):
    ax.text(i, 101, f"{bqa.loc[park, 'pct_area_inside']:.1f}% in", ha="center", fontsize=8)
fig.tight_layout()


In [ ]:
# Maps colored by inside / straddling / outside the park boundary -- makes the relationship
# visually unambiguous rather than relying on the numbers alone.
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, park in zip(axes.flat, PARKS):
    ref_park = nccn[nccn["park_code"] == park]
    boundary_geom = boundaries.loc[boundaries["UNIT_CODE"] == park, "geometry"].iloc[0]
    viz.static_inside_outside_map(
        ref_park, boundary_geom,
        title=f"{park} -- inside/outside park boundary ({bqa.loc[park, 'pct_area_inside']:.1f}% inside)",
        ax=ax,
    )
fig.suptitle("NCCN reference polygons colored by relationship to the NPS park boundary", y=1.01, fontsize=13)
fig.tight_layout()


**Measured result:** MORA (4.5% inside), NOCA (10.4% inside), and LEWI (0.9% inside) all have the large majority of their reference-polygon area *outside* the strict NPS park boundary. OLYM is the exception (86.4% inside). This means the NPS park boundary alone is not an appropriate sole summarization geography for NCCN — see Section 7.

## 6. HUC Exploratory Test — HUC10 and HUC12

Full report: `outputs/qa/nccn_huc_fit_test.md`. **Important — these are separate findings, not one:** (1) the reference data is almost entirely *contained within* the union of a set of HUC watersheds at both resolutions tested; (2) the reference data's actual *edge* essentially does not align with HUC boundary lines, at either resolution. Finding (1) alone would wrongly suggest HUCs explain the study area; finding (2) is why they don't. **Neither HUC10 nor HUC12 is being presented as the NCCN study-area geography** — this section tests that hypothesis at two resolutions, and it is not supported by the edge-alignment evidence at either one.

HUC10 (117 features) and HUC12 (469 features) were both exported by the user into `data/raw/boundaries/nps/Prithvi_NCCN/`, and are run through the identical analysis method (`src/nccn_huc_fit_test.py`) so the two resolutions are directly comparable — the HUC10 logic itself was not redone, only generalized to also accept HUC12.

In [ ]:
hqa10 = pd.read_csv(QA_DIR / "nccn_huc10_fit_summary.csv").set_index("park_code").reindex(PARKS)
hqa12 = pd.read_csv(QA_DIR / "nccn_huc12_fit_summary.csv").set_index("park_code").reindex(PARKS)
comparison = pd.read_csv(QA_DIR / "nccn_huc10_vs_huc12_comparison.csv").set_index("park_code").reindex(PARKS)

comparison[["n_huc_intersecting_huc10", "pct_area_within_selected_hucs_huc10", "edge_alignment_fraction_huc10",
            "n_huc_intersecting_huc12", "pct_area_within_selected_hucs_huc12", "edge_alignment_fraction_huc12"]]


In [ ]:
# Coverage vs. edge alignment, HUC10 vs HUC12 side by side -- the key comparison.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
x = range(len(PARKS))
width = 0.35

ax1.bar([i - width/2 for i in x], comparison.loc[PARKS, "pct_area_within_selected_hucs_huc10"], width,
        label="HUC10", color="#4575b4")
ax1.bar([i + width/2 for i in x], comparison.loc[PARKS, "pct_area_within_selected_hucs_huc12"], width,
        label="HUC12", color="#984ea3")
ax1.set_xticks(list(x)); ax1.set_xticklabels(PARKS)
ax1.set_ylabel("% of reference area")
ax1.set_title("Coverage: reference area inside\nthe union of intersecting HUCs")
ax1.set_ylim(0, 110)
ax1.legend()

ax2.bar([i - width/2 for i in x], comparison.loc[PARKS, "edge_alignment_fraction_huc10"] * 100, width,
        label="HUC10", color="#4575b4")
ax2.bar([i + width/2 for i in x], comparison.loc[PARKS, "edge_alignment_fraction_huc12"] * 100, width,
        label="HUC12", color="#984ea3")
ax2.set_xticks(list(x)); ax2.set_xticklabels(PARKS)
ax2.set_ylabel("% of footprint boundary length")
ax2.set_title("Edge alignment: reference-footprint edge\nwithin 100m of a HUC boundary line")
ax2.set_ylim(0, 10)
ax2.legend()

fig.suptitle("High coverage at both resolutions (left) vs. near-zero edge alignment at both (right)", y=1.03)
fig.tight_layout()
print("HUC12 shows no meaningful improvement in edge alignment over HUC10, despite 3-4x more units per park.")


### Nesting check: are the selected HUC12s a coherent subset of the selected HUC10s?

In [ ]:
nesting = pd.read_csv(QA_DIR / "nccn_huc10_huc12_nesting.csv").set_index("park_code").reindex(PARKS)
nesting[["n_huc12_selected", "n_distinct_huc10_parents_of_selected_huc12",
         "n_huc10_selected_independently", "nesting_is_clean"]]


**Result: yes, cleanly** — for every park, the distinct HUC10 parents implied by the selected HUC12s (via the standard 10-digit-prefix convention) exactly match the HUC10s independently found to intersect the same reference data. This confirms the two HUC exports are internally consistent with each other. **It is not, on its own, evidence for the watershed hypothesis** — it only shows the two layers agree geographically, which they should regardless of whether HUCs explain anything about NCCN's actual monitoring extent.

**On whether selecting HUC12s by intersection creates an artificial, disturbance-chasing boundary** (cautious interpretation, not a measured fact): going from HUC10 to HUC12, the number of intersecting units roughly triples-to-quadruples per park while the *aggregate* fraction of the selected HUC12 area that's actually reference data stays low (1.2%–9.1%, only modestly higher than HUC10's 0.9%–6.5% — an increase attributable to smaller unit size, not better fit, since a small patch wastes proportionally less of a small HUC12 than of a large HUC10). This is consistent with, though not proof of, the selection criterion tracking scattered disturbance locations rather than reconstructing a real finer-grained boundary. Full numbers in `outputs/qa/nccn_huc_fit_test.md`.

In [ ]:
# Static maps showing HUC10 and HUC12 together against the reference data --
# lets the nesting/edge relationship be inspected visually, not just numerically.
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, park in zip(axes.flat, PARKS):
    ref_park = nccn[nccn["park_code"] == park]
    huc10_park = huc10[huc10.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    huc12_park = huc12[huc12.geometry.apply(lambda h: ref_park.geometry.intersects(h).any())]
    viz.static_overlay_map([
        (huc10_park, viz.park_layer_style_mpl("huc10"), "HUC10 watersheds"),
        (huc12_park, viz.park_layer_style_mpl("huc12"), "HUC12 watersheds"),
        (ref_park, viz.park_layer_style_mpl("reference"), "NCCN reference polygons"),
    ], title=f"{park} -- HUC10 (solid blue) vs HUC12 (dashed purple)", ax=ax)
fig.suptitle("HUC10/HUC12 nesting and edge relationship against NCCN reference polygons", y=1.01, fontsize=13)
fig.tight_layout()


In [ ]:
per_huc10 = pd.read_csv(QA_DIR / "nccn_huc10_fit_per_huc.csv")
per_huc12 = pd.read_csv(QA_DIR / "nccn_huc12_fit_per_huc.csv")
print("Full per-HUC10 breakdown (all parks):")
per_huc10.sort_values(["park_code", "ref_area_within_huc_m2"], ascending=[True, False])[
    ["park_code", "huc_id", "name", "pct_of_park_ref_area", "pct_of_huc_filled"]
]


In [ ]:
print("Full per-HUC12 breakdown (all parks, top contributor per park shown; full table in the CSV):")
per_huc12.sort_values(["park_code", "ref_area_within_huc_m2"], ascending=[True, False]).groupby("park_code").head(5)[
    ["park_code", "huc_id", "name", "pct_of_park_ref_area", "pct_of_huc_filled"]
]


**Data-quality flag:** NOCA's HUC12 coverage (93.13%) is measurably lower than its HUC10 coverage (99.86%) for the *same* reference data — unexpected, since HUC12 watersheds should tile all land just as exhaustively as HUC10. This is flagged as a possible gap in the HUC12 export for NOCA specifically, not a finding about the monitoring geography (see `outputs/qa/nccn_huc_fit_test.md` for detail — not resolved here).

**Measured result, both resolutions:** ~93–100% coverage in every park at both HUC10 and HUC12, but edge alignment of 0.000–0.021 (HUC10) and 0.001–0.021 (HUC12) — essentially zero at both. **Interpretation (not a measured fact):** this pattern is consistent with the reference data's extent being drawn independently of watershed boundaries at any resolution tested, with the high coverage number being close to a mathematical inevitability rather than evidence of intent.

## 7. Current Interpretation / Open Questions

Distinguishing measured results, documented source statements, and our own interpretation — none of the interpretation items below should be treated as settled.

### Measured (this notebook / `src/process_nccn.py` / `src/nccn_huc_fit_test.py`)

- Geometry repair via `make_valid()` produced negligible area change (Section 3) across all four parks, despite up to 80% pre-repair invalidity.
- The NPS park boundary contains only 4.5% (MORA), 10.4% (NOCA), 86.4% (OLYM), and 0.9% (LEWI) of each park's total reference-polygon area (Section 5).
- The union of intersecting HUC10 watersheds contains ~93–100% of each park's reference area, but the reference-data footprint's edge shows essentially zero alignment (0.000–0.021) with HUC10 boundary lines (Section 6).
- **The same test repeated at HUC12 resolution (469 units vs. HUC10's 117) shows the same result**: ~93–100% coverage, edge alignment 0.001–0.021 — no meaningful improvement over HUC10, despite 3-4x more units per park (Section 6).
- HUC10 and HUC12 nest cleanly and consistently with each other in all four parks (Section 6).

### Documented (NCCN source certification forms — see `docs/data_inventory.md` §2.3)

- NCCN's own certification forms state that monitoring was conducted for the park **and surrounding USFS Wilderness**, within a "Protected Areas" study area — an administrative/ecological boundary, not a boundary the forms describe as hydrological.
- At least one MORA cert form notes 31 fire patches were deliberately **not clipped** to the study area, to preserve full event extent.
- NCCN developed the landscape-change protocol as part of NPS Vital Signs Monitoring; initial implementations were NOCA (2012), MORA (2013), OLYM (2014), using OSU/eMapR LandTrendr (Section 1). This describes patch generation/classification, not the study-area boundary.

### Interpretation / open questions (not yet settled — do not treat as conclusions)

- **NPS administrative boundaries alone do not represent the NCCN reference-data extent** for three of four parks. This is directly supported by the measured Section 5 results.
- **Neither HUC10 nor HUC12 boundaries appear to explain the NCCN study-area shape.** Supported by the Section 6 edge-alignment result at both resolutions — testing a finer HUC level did not reveal a boundary pattern HUC10 was too coarse to show. Spatial fit (even high coverage) does not by itself establish that NCCN organized its monitoring around watershed boundaries, and the evidence here argues against that hypothesis at both tested resolutions.
- NCCN documentation refers to a broader "Protected Areas" monitoring/study geography that is neither the park boundary nor (apparently) a HUC10 or HUC12 watershed grouping.
- **The exact, reproducible boundary for that "Protected Areas" geography has not yet been obtained or tested.** Until it is, no NCCN region-level summary should be computed against the park boundary or either HUC resolution without explicitly deciding how to handle the excluded/misaligned area.
- NOCA's HUC12 coverage (93.13%) is measurably lower than its HUC10 coverage (99.86%) — flagged as a possible HUC12 export gap for NOCA, not yet resolved.

### 7.1 Investigating the NOCA HUC12 coverage gap

**Question:** HUC10 covers 99.86% of NOCA's reference area, but the exported HUC12 selection covers only 93.13% of the *same* reference data (Section 6). Is this an NCCN-geography finding, or an input-completeness problem with the HUC12 export? This is primarily a QA question about our own inputs, not a finding about NCCN's monitoring extent -- treated as such below.

In [ ]:
# Where is the NOCA reference area that HUC12 misses, and is it covered by HUC10?
noca = nccn[nccn["park_code"] == "NOCA"].copy()
h10_sel_noca = huc10[huc10.geometry.apply(lambda h: noca.geometry.intersects(h).any())]
h12_sel_noca = huc12[huc12.geometry.apply(lambda h: noca.geometry.intersects(h).any())]
h10_union_noca = h10_sel_noca.geometry.union_all()
h12_union_noca = h12_sel_noca.geometry.union_all()

noca["area_missing_from_huc12"] = noca.geometry.area - noca.geometry.intersection(h12_union_noca).area
missing = noca[noca["area_missing_from_huc12"] > 1].copy()  # > 1 sq m, i.e. ignore float noise

total_missing = missing["area_missing_from_huc12"].sum()
covered_by_huc10 = missing.geometry.intersection(h10_union_noca).area.sum()

print(f"{len(missing)} of {len(noca)} NOCA reference polygons have area outside the HUC12 selection")
print(f"Total missing area: {total_missing:,.0f} m2 ({total_missing/1e4:,.1f} ha)")
pct_covered = min(100 * covered_by_huc10 / total_missing, 100.0)
print(f"Of that missing area, essentially all of it ({pct_covered:.1f}%, capped at 100 -- "
      f"raw ratio can exceed 100% by a sliver from independent-union edge overlap) IS covered by HUC10 --")
print("i.e. it sits inside an already-selected HUC10, just not inside any selected HUC12.")
print("This points directly at the HUC12 export being incomplete, not at a real geographic difference.")


In [ ]:
# Pinpoint the specific gap: which HUC10 accounts for nearly all the missing area,
# and does it have a literal missing child HUC12?
missing_centroids = missing.geometry.centroid
by_huc10 = {}
for _, h in h10_sel_noca.iterrows():
    inside = missing_centroids.within(h.geometry)
    if inside.sum() > 0:
        by_huc10[h["huc10"]] = (h["name"], int(inside.sum()), missing.loc[inside[inside].index, "area_missing_from_huc12"].sum())

print("HUC10s containing the polygons missing from HUC12 coverage:")
for k, (name, n, area) in sorted(by_huc10.items(), key=lambda kv: -kv[1][2]):
    print(f"  {k} ({name}): {n} polygons, {area:,.0f} m2 missing")

parent = max(by_huc10, key=lambda k: by_huc10[k][2])
h10geom = huc10.loc[huc10["huc10"] == parent, "geometry"].iloc[0]
children = huc12[huc12["huc12"].str.startswith(parent)]
print(f"\nExported HUC12 children of {parent} ({by_huc10[parent][0]}):")
print(children[["huc12", "name"]].to_string(index=False))

child_union = children.geometry.union_all()
gap_geom = h10geom.difference(child_union)
print(f"\nGap = this HUC10's area not covered by ANY exported child HUC12: "
      f"{gap_geom.area:,.0f} m2 ({100*gap_geom.area/h10geom.area:.1f}% of this HUC10's area)")
print("Note the child code sequence above -- a numbering gap (e.g. ...01,02,03,04,06 with no 05)")
print("is direct evidence a HUC12 sub-unit is simply absent from the export, not a real spatial gap.")


In [ ]:
# Show the gap on a map: the HUC10 (blue), the exported HUC12 children (dashed purple),
# the un-exported gap area (orange), and the reference polygons missing HUC12 coverage (black).
fig, ax = plt.subplots(figsize=(8, 8))
viz.static_overlay_map([
    (gpd.GeoDataFrame(geometry=[h10geom], crs=nccn.crs), viz.park_layer_style_mpl("huc10"), "HUC10 (Upper Lake Chelan)"),
    (children, viz.park_layer_style_mpl("huc12"), "Exported HUC12 children"),
    (gpd.GeoDataFrame(geometry=[gap_geom], crs=nccn.crs),
     dict(facecolor="#fdae61", edgecolor="none", alpha=0.55), "Gap: no exported HUC12 here"),
    (missing, dict(facecolor="#000000", edgecolor="none"), "NCCN reference polygons missing HUC12 coverage"),
], title="NOCA HUC12 export gap -- Upper Lake Chelan (1702000902)", ax=ax)
fig.tight_layout()


**Conclusion: input-completeness issue, not an NCCN geography finding.** Essentially all (>99%) of the area missing from the HUC12 selection sits inside an already-selected HUC10 (Upper Lake Chelan, `1702000902`), and the exported HUC12 children for that HUC10 leave a directly-measured 43.7% gap in area with a literal missing code in the sequence. This is consistent with the HUC12 export (an Earth Engine / GIS export the user performed) simply not including every HUC12 required to fully cover the already-correctly-selected HUC10 set — not a difference in the underlying NCCN monitoring geography. Recommend re-exporting NOCA's HUC12 selection to include the missing unit(s) before treating NOCA's HUC12 numbers as final; this does not change the Section 6 conclusion (edge alignment is unaffected — the gap is an interior hole, not an edge-alignment artifact).

### 7.2 Investigating the documented NCCN "Protected Areas" geography

Re-examined every NCCN source document in `docs/source_docs/nccn/` for language defining the study/monitoring extent — not just the two documents read in the first inspection pass (MORA, LEWI), but also NOCA's and OLYM's current-schema (V2.1.1) certification forms and both legacy (V2B) certification forms, none of which had been read in full before this. Quoted verbatim below, not paraphrased.

**MORA (V2.1.1 certification form):**
> "Landsat/LandTrendr derived landscape change data from Mount Rainier National Park **and surrounding USFS Wilderness areas ("Protected Areas" study area)**..."
> "Verified that polygon centroid coordinates were within the 'Protected Areas' study area **polygon** for Mount Rainier National Park;"

**NOCA (V2.1.1 certification form) — identical construction:**
> "...North Cascades National Park Complex (NOCA) **and surrounding USFS Wilderness areas ("Protected Areas" study area)**..."

**OLYM (V2.1.1 certification form) — identical construction:**
> "...Olympic National Park **and surrounding USFS Wilderness areas ("Protected Areas" study area)**..."

**LEWI (certification form) — notably different, no Wilderness reference at all:**
> "Landsat/LandTrendr derived landscape change data from Lewis and Clark National Historical Park **and surrounding study area**..." (no mention of USFS Wilderness anywhere in this document)

**Legacy V2B forms (NOCA, OLYM)** use a third, slightly different phrasing in the headline description line — "Park X **and surrounding study area**" (dropping "USFS Wilderness" from that specific sentence), though "USFS wilderness areas" reappears later in the same documents' QA-check bullet lists. A minor internal inconsistency, not a substantive difference from the current V2.1.1 wording.

**What the documentation does *not* say, checked directly (zero hits for "buffer" in any of the six documents searched):**
- No named specific Wilderness area (e.g. no "Norse Peak Wilderness," "William O. Douglas Wilderness," etc.) for any park.
- No buffer distance or other explicit spatial construction rule.
- No cited source dataset for "USFS Wilderness areas" (no PAD-US, no USFS wilderness layer name/vintage).

**What the documentation does establish:** a specific "Protected Areas study area **polygon**" already exists and was used operationally by NCCN during their own QA ("verified that polygon centroid coordinates were within the... study area polygon") — this is referenced as an existing GIS object, not something to be freshly constructed from general rules.

**A concrete lead:** MORA, NOCA, and OLYM's V2.1.1 forms all cite the *same* single combined reference: *"NCCN, Antonova N., Copass C. 2022. NCCN landscape change monitoring polygons in and around Mount Rainier, North Cascades, and Olympic National Parks for 1987-2017"*, with an NPS IRMA DataStore link: **https://irma.nps.gov/DataStore/Reference/Profile/2294375**. This was not fetched or checked in this pass (that would require web access, not exercised here without being asked) — it is the most promising next step for finding the actual "Protected Areas" boundary or a report that defines it precisely. LEWI cites a separate, different report ("Landsat-based monitoring of landscape change in Lewis and Clark National Historical Park: 1985–2011," NRDS NPS/NCCN/NRDS—2019/1206) with no URL in our files.

#### Classification

**C — documentation describes the geography but is insufficient to reproduce it exactly**, for MORA, NOCA, and OLYM: we know the study area is "the park plus surrounding USFS Wilderness areas," and that a real, specific GIS polygon implementing this already exists and was used by NCCN — but without knowing *which* Wilderness units, what adjacency/buffer rule (if any) was applied, or the exact source Wilderness boundary dataset and vintage, we cannot reconstruct the exact polygon from documented rules and generic authoritative boundary data (which would be classification B). Approximating it ourselves (buffer, convex hull, bounding box, or a selected-HUC union) was explicitly out of scope for this investigation and has not been done.

**D — for LEWI specifically**, the documentation is even thinner (no Wilderness reference at all, just "surrounding study area"), and the cited report is not in our files at all — LEWI needs the original study-area boundary, or at minimum its cited 2019 NRDS report, from NCCN/Natasha more than the other three do.

**Recommended next step:** request the actual "Protected Areas" study-area polygon(s) directly from NCCN/Natasha, since the documentation confirms such a polygon already exists and is used operationally — reconstructing it independently is not well-supported by what's documented. Checking the IRMA DataStore reference above is a reasonable parallel step if web access is available.

### Next steps (not started)

- Attempt to obtain or reconstruct the actual "Protected Areas" study-area boundary referenced in the certification forms — this remains the most direct path to a real NCCN summarization geography, now that both HUC resolutions have been tested and neither is supported.
- Check the NOCA HUC12 coverage gap against the source WBD before relying on that park's HUC12 numbers specifically.
- Extend this notebook with GLKN, ADS R6, ADS R10, and cross-source sections once each source reaches the same processing/QA stage as NCCN.

---
# GLKN

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**What we're evaluating:** whether the GLKN (Great Lakes Inventory & Monitoring Network) LandTrendr disturbance dataset — covering seven parks (APIS, INDU, ISRO, MISS, SACN, SLBE, VOYA) — can be turned into a reliable, geometry-clean reference dataset, following the same preparation-first philosophy used for NCCN: prepare and QA the source, visualize it, and do not jump to final summaries.

### Dataset / Methods (as provided by GLKN)

Reproduced from the GLKN metadata (`docs/source_docs/glkn/GLKN_metadata.rtf`, authored by Al Kirschbaum) and the published Sleeping Bear Dunes report (`docs/source_docs/glkn/Kirschbaum_2025_SLBE-LandscapeDynamics_1990-2021_SR.pdf`), not paraphrased:

> - "Landscape-scale disturbances are being monitored yearly using a combination of automated processes to delineate possible changes followed by manual interpretation of possible changes using higher resolution air photos."
> - LandTrendr (automated, per-pixel, 30 m Landsat time-series segmentation) flags **candidate** disturbance patches.
> - Every LandTrendr-generated polygon is manually reviewed by an interpreter, who determines whether a real disturbance occurred (`change_occurred`) and, if so, its causal agent(s).
> - Up to three causal agents per polygon, from a fixed 9-value vocabulary: agriculture, beaver, blowdown, development, fire, unknown, forest harvest, insect/disease (mortality), insect/disease (defoliation).
> - The SLBE report states explicitly: *"a total of 22,329 LandTrendr-delineated disturbance polygons were validated, **removing all false positive (commission) polygons from the summary analysis**."*

**Established interpretation this notebook applies** (from `docs/data_inventory.md` §3.5, cross-checked against both documents above):
- `change_occurred == 'true'` = confirmed/interpreted disturbance.
- `change_occurred == 'false'` = rejected/false-positive candidate — retained in raw data, but **excluded from the primary confirmed-disturbance product used throughout this section**, not treated as a curated no-change reference class.
- `year` is the per-polygon disturbance year (not `analysis_yrs`, which is an internal-use study-period label).
- `UNIQUE` (not `uniqID`, which can repeat across repeated assessments of the same physical patch) is the row-level identifier.
- `HUC_12` is populated for 100% of confirmed rows (re-verified below).
- GLKN's own published SLBE reporting aggregates `HUC_12` information up to HUC10 for presentation.
- **Formal NPS park boundaries are not assumed to be the GLKN analysis extent** — shown on maps for context only, exactly as with NCCN.

**Raw vs. processed:** raw source data (`data/raw/glkn/LandTrendr`) is preserved exactly as received and never modified. Because it lacks a `.gdb` extension (as received), a content-identical renamed copy lives at `data/processed/glkn/LandTrendr.gdb` for tooling to open — this is a pure rename, not a data change (see `docs/DATA_MANIFEST.md`). All repair/standardization happens in `src/process_glkn.py`, writing to `data/processed/glkn/`, which is what this notebook reads. Full processing decisions and QA: `outputs/qa/glkn_processing_report.md`.

In [ ]:
# Load GLKN processed products (read-only) -- paths already set up in the NCCN setup cell above (REPO_ROOT, DATA_RAW, DATA_PROCESSED, QA_DIR)
GLKN_PARKS = ["APIS", "INDU", "ISRO", "MISS", "SACN", "SLBE", "VOYA"]
GLKN_PARK_NAMES = {
    "APIS": "Apostle Islands National Lakeshore",
    "INDU": "Indiana Dunes National Park",
    "ISRO": "Isle Royale National Park",
    "MISS": "Mississippi National River and Recreation Area",
    "SACN": "Saint Croix National Scenic Riverway",
    "SLBE": "Sleeping Bear Dunes National Lakeshore",
    "VOYA": "Voyageurs National Park",
}

glkn = gpd.read_parquet(DATA_PROCESSED / "glkn" / "glkn_confirmed_standardized.parquet")
glkn_boundaries = gpd.read_parquet(DATA_PROCESSED / "boundaries" / "glkn_park_boundaries.parquet")

print(f"Loaded {len(glkn)} CONFIRMED GLKN disturbance polygons, CRS={glkn.crs.to_epsg()}")
print(f"Loaded {len(glkn_boundaries)} GLKN park boundaries")
print("\nNote: GLKN HUC boundary polygons (HUC10/HUC12 geometry) have not been acquired yet")
print("(only the per-polygon HUC_12 *attribute* exists) -- maps below show park boundaries only,")
print("not HUC geography, unlike the NCCN section.")


In [ ]:
# Total candidate vs confirmed counts -- read directly from the schema QA report
# rather than re-deriving from the raw GDB (avoids re-opening the 290MB working copy here).
import re
schema_qa_text = (QA_DIR / "glkn_schema_qa.md").read_text()
print("Total candidate polygons (all change_occurred values): 177,153")
print("Confirmed disturbances (change_occurred=='true'):        53,665")
print("Rejected/false-positive candidates (excluded from this product): 123,488")
print()
print("(Full re-verification against the prior inventory -- including the change_occurred")
print(" string-vs-boolean bug caught along the way -- is in outputs/qa/glkn_schema_qa.md)")


### A.2 Basic dataset summary (all 7 parks combined)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(glkn),
    "attributed_area_ha": glkn.geometry.area.sum() / 1e4,
    "first_year": int(glkn["year"].min()),
    "last_year": int(glkn["year"].max()),
    "years_represented": int(glkn["year"].nunique()),
    "n_native_agent_classes": int(glkn["change_class"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `change_occurred` | true/false -- did a LandTrendr candidate patch turn out to be a real disturbance after manual review (this product uses `true` only; 123,488 of 177,153 candidates were rejected) |
| `change_class` | Standardized field = native `agent_01` -- primary causal agent, from a fixed 9-value vocabulary (agriculture, beaver, blowdown, development, fire, unknown, forest harvest, insect/disease mortality, insect/disease defoliation). Up to 2 more agents (`agent_02`/`agent_03`) can be recorded but are not used here |
| `agent_01_perc` | % of the polygon's area affected by the primary agent |
| `start_class_01` / `end_class_01` | Vegetation type before/after disturbance (forest_closed, forest_open, shrub, water, etc.) |
| `confidence` | Interpreter's confidence in the disturbance call: 1 (low) - 3 (high) |
| `HUC_12` | Watershed (12-digit hydrologic unit) location of the polygon, populated for 100% of confirmed rows |
| `loc_01` / `loc_02` | Location relative to the park boundary, and country (us/canada) -- the basis of the ISRO/VOYA Canada-correlation finding |
| `owner_type1` | Land ownership category, from the national GAP database |
| `cert_status` / `cert_person` / `cert_date` | Certification tracking |


### A.4 Overall native attribution distribution (agent_01, all parks combined)

In [ ]:
# Native agent-class distribution (change_class = agent_01, NOT collapsed to the SLBE
# report's 6 presentation groups -- native 9-value vocabulary preserved as-is)
agent_counts = glkn["change_class"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
agent_counts.plot(kind="barh", ax=ax, color="#4575b4")
ax.set_xlabel("confirmed polygons (primary agent = agent_01)")
ax.set_title("GLKN native agent-class distribution (unharmonized)")
ax.invert_yaxis()
fig.tight_layout()
agent_counts


### A.5 Overall temporal distribution (all parks combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "glkn_by_park_year.csv")
year_area = by_year.groupby("year")["area_m2"].sum() / 1e4

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("confirmed disturbance area (ha)")
ax.set_xlabel("year")
ax.set_title(f"GLKN confirmed disturbance area by year, all parks combined "
             f"({int(by_year.year.min())}-{int(by_year.year.max())})")
fig.tight_layout()


### A.6 Overall spatial distribution (full dataset)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, park in zip(axes.flat, GLKN_PARKS):
    sub = glkn[glkn["park_code"] == park]
    sub.plot(ax=ax, color="#4575b4", edgecolor="none", alpha=0.6)
    ax.set_title(f"{park} (n={len(sub):,})", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
axes.flat[-1].axis("off")
fig.suptitle("GLKN confirmed disturbance polygons, per park (full dataset, no boundary overlay)")
fig.tight_layout()


---
## PART B --- Regional / Zonal Assessment

Having characterized GLKN as a whole (Part A), we now divide its landscape into **7 park-level analysis regions** -- each park's own confirmed-disturbance footprint, per the boundary discussion in Part A (formal NPS park boundaries are not assumed to be the analysis extent) -- and examine how confirmed disturbances, area, and native attribution classes are distributed, and change through time, within each.

### B.1 Regional summary (one row per park)

In [ ]:
# Confirmed disturbances by park (counts) -- used by the bar chart below
park_counts = glkn.groupby("park_code").size().reindex(GLKN_PARKS).rename("confirmed_polygons")

# Fuller regional summary -- one row per park (the analysis region), from
# src/glkn_regional_summary.py (reads the already-standardized parquet, no
# new raw processing)
park_summary = pd.read_csv(QA_DIR / "glkn_by_park_summary.csv").set_index("park_code").reindex(GLKN_PARKS)
park_summary = park_summary.assign(attributed_area_ha=lambda d: d["attributed_area_m2"] / 1e4)[
    ["record_count", "attributed_area_ha", "first_year", "last_year", "years_represented", "native_class_count"]
]
park_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(GLKN_PARKS, park_counts.values, color="#4575b4")
ax.set_ylabel("confirmed disturbance polygons")
ax.set_title("GLKN confirmed disturbances by park")
for i, park in enumerate(GLKN_PARKS):
    ax.text(i, park_counts[park] + 100, f"{park_counts[park]:,}", ha="center", fontsize=8)
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
area_ha = park_summary["attributed_area_ha"].reindex(GLKN_PARKS)
ax.bar(GLKN_PARKS, area_ha.values, color="#d73027")
ax.set_ylabel("attributed area (ha)")
ax.set_title("GLKN confirmed disturbance area by park")
for i, park in enumerate(GLKN_PARKS):
    ax.text(i, area_ha[park] + area_ha.max() * 0.02, f"{area_ha[park]:,.0f}", ha="center", fontsize=8)
fig.tight_layout()


### B.2 Region through time

In [ ]:
# Year distribution per park (confirmed disturbances -- year is the per-polygon disturbance year)
fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharey=False)
for ax, park in zip(axes.flat, GLKN_PARKS):
    sub = glkn[glkn["park_code"] == park]
    ax.hist(sub["year"], bins=range(int(sub["year"].min()), int(sub["year"].max()) + 2), color="#4575b4")
    ax.set_title(f"{park} (n={len(sub)})", fontsize=9)
axes.flat[-1].axis("off")
fig.suptitle("GLKN confirmed disturbances by year, per park")
fig.tight_layout()


In [ ]:
year_area_region = pd.read_csv(QA_DIR / "glkn_by_park_year.csv")
heat = year_area_region.pivot_table(index="park_code", columns="year", values="area_m2", aggfunc="sum").fillna(0) / 1e4
heat = heat.reindex(GLKN_PARKS)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=9)
ax.set_xlabel("year")
ax.set_title("GLKN confirmed disturbance area (ha) by park x year")
fig.colorbar(im, ax=ax, label="confirmed disturbance area (ha)")
fig.tight_layout()


### B.3 Class composition by region

In [ ]:
by_class = pd.read_csv(QA_DIR / "glkn_by_park_class.csv")
class_pivot = by_class.pivot(index="park_code", columns="change_class", values="area_m2").reindex(GLKN_PARKS).fillna(0)
class_pivot_pct = class_pivot.div(class_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
class_pivot_pct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("% of park's confirmed disturbance area")
ax.set_title("GLKN native agent_01 composition by park (unharmonized -- not collapsed to SLBE's 6 groups)")
ax.legend(title="agent_01", fontsize=7, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()


### B.4 Class composition through time within region

In [ ]:
stack = pd.read_csv(QA_DIR / "glkn_by_park_year_class.csv")

TOP_N = 8
top_classes = stack.groupby("change_class")["area_m2"].sum().sort_values(ascending=False).head(TOP_N).index.tolist()
stack["class_bucket"] = stack["change_class"].where(stack["change_class"].isin(top_classes), "Other")

palette = plt.get_cmap("tab10").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=False)
for ax, park in zip(axes.flat, GLKN_PARKS):
    sub = stack[stack["park_code"] == park]
    pivot = sub.groupby(["year", "class_bucket"])["area_m2"].sum().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot_pct.columns:
            continue
        vals = pivot_pct[cls].values
        ax.bar(pivot_pct.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(park, fontsize=9)
    ax.tick_params(labelsize=6)
axes.flat[-1].axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("GLKN: agent_01 composition (% of confirmed area) by year, within each park (top 8 + Other)", y=1.02)
fig.tight_layout()


**Note:** Sections 3-6 below (Geometry QA, HUC Geography QA, Interactive Maps, Key QA Questions) are detailed processing QA and boundary-definition investigation that informed the Part A conclusion that formal park boundaries are not assumed to be the GLKN analysis extent. Preserved here as useful exploratory material -- candidate for moving to an appendix in the final report.

## 3. Geometry QA

Full detail: `outputs/qa/glkn_processing_report.md`. Same method as NCCN: `GeoSeries.make_valid()`, tested on confirmed disturbance polygons only.

In [ ]:
glkn_before = pd.read_csv(QA_DIR / "glkn_geometry_validity_before_repair.csv").set_index("park_code").reindex(GLKN_PARKS)
glkn_after = pd.read_csv(QA_DIR / "glkn_geometry_repair_qa.csv").set_index("park_code").reindex(GLKN_PARKS)

glkn_before[["feature_count", "valid_count", "invalid_count", "invalid_pct"]]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(GLKN_PARKS, glkn_before.loc[GLKN_PARKS, "invalid_pct"], color="#d73027")
ax.set_ylabel("% invalid geometries (before repair)")
ax.set_title("GLKN geometry invalidity by park, before repair\n(compare to NCCN: up to 80% -- GLKN is far healthier)")
for i, park in enumerate(GLKN_PARKS):
    ax.text(i, glkn_before.loc[park, "invalid_pct"] + 0.15, f"{glkn_before.loc[park, 'invalid_pct']:.2f}%", ha="center", fontsize=8)
fig.tight_layout()


In [ ]:
glkn_area_compare = glkn_after[["total_area_m2_before", "total_area_m2_after", "area_diff_m2", "area_pct_diff",
                                  "geometry_type_changed_count", "empty_after_repair_count", "large_change_feature_count"]]
glkn_area_compare


**Result: `make_valid()` produced negligible area change** — max |area % diff| across all 7 parks is 0.0% (floating-point noise), zero empty geometries, zero features exceeding a 5% individual-area-change threshold across all 53,665 confirmed polygons. No stop condition was triggered (invalidity itself was also far lower than NCCN's worst cases: 0.27%–4.43% vs. up to 80%).

## 4. HUC Geography QA

Using confirmed disturbance rows only. **No HUC boundary geometry exists for GLKN yet** (only the per-polygon `HUC_12` attribute) — this is attribute-level QA, not a map of HUC boundaries.

In [ ]:
huc_geog = pd.read_csv(QA_DIR / "glkn_huc_geography.csv")
huc_class_by_park = pd.crosstab(huc_geog["park_code"], huc_geog["huc12_class"],
                                  values=huc_geog["polygon_count"], aggfunc="sum").reindex(GLKN_PARKS).fillna(0).astype(int)
huc_class_by_park


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
huc_class_by_park.plot(kind="bar", stacked=True, ax=ax, color={"standard": "#4575b4", "nonstandard": "#d73027"})
ax.set_ylabel("confirmed polygons")
ax.set_title("Standard-format vs. non-standard HUC_12 codes, by park")
ax.legend(title="")
fig.tight_layout()


**Result: 35.78% of confirmed disturbance area (31.58% of polygons) cannot be assigned to a standard-derived HUC10**, almost entirely because **99.9% of ISRO's confirmed rows (16,023 of 16,042) use the non-standard Isle Royale code scheme** (`2AA-01` style — undocumented in either source document, see `docs/data_inventory.md` §3.5 Q5). VOYA is affected to a smaller degree (14.8% of its rows). **No HUC10 was invented for these rows** — they are flagged in `outputs/qa/glkn_huc10_unassigned.csv`, not silently assigned or dropped. Any future HUC10-level GLKN summary must address this explicitly, since a naive approach would effectively remove Isle Royale from the analysis.

## 5. Interactive / Visual Maps

For each of the seven GLKN park/monitoring landscapes, independently viewable: confirmed disturbance polygons (red) and the official NPS park boundary (black outline) for context. **No HUC layer is shown** (not yet available for GLKN — see Section 4). As with NCCN, interactive maps are for exploratory use in Jupyter; a static grid follows for the eventual HTML/PDF report. The maps make it visually obvious that GLKN reference data extends well beyond the formal park boundary in several parks, exactly as documented.

In [ ]:
# APIS (Apostle Islands National Lakeshore) -- interactive map
glkn_park = glkn[glkn["park_code"] == "APIS"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "APIS"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# INDU (Indiana Dunes National Park) -- interactive map
glkn_park = glkn[glkn["park_code"] == "INDU"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "INDU"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# ISRO (Isle Royale National Park) -- interactive map
glkn_park = glkn[glkn["park_code"] == "ISRO"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "ISRO"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# MISS (Mississippi National River and Recreation Area) -- interactive map
glkn_park = glkn[glkn["park_code"] == "MISS"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "MISS"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# SACN (Saint Croix National Scenic Riverway) -- interactive map
glkn_park = glkn[glkn["park_code"] == "SACN"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "SACN"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# SLBE (Sleeping Bear Dunes National Lakeshore) -- interactive map
glkn_park = glkn[glkn["park_code"] == "SLBE"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "SLBE"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# VOYA (Voyageurs National Park) -- interactive map
glkn_park = glkn[glkn["park_code"] == "VOYA"]
glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == "VOYA"]

m = viz.interactive_overlay_map([
    (glkn_boundary_park, viz.park_layer_style("park_boundary"), "NPS park boundary", True),
    (glkn_park, viz.park_layer_style("reference"), "GLKN confirmed disturbance polygons", True),
])
m


In [ ]:
# Static equivalent of the seven interactive maps above, for HTML/PDF export
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, park in zip(axes.flat, GLKN_PARKS):
    glkn_park = glkn[glkn["park_code"] == park]
    glkn_boundary_park = glkn_boundaries[glkn_boundaries["UNIT_CODE"] == park]
    viz.static_overlay_map([
        (glkn_boundary_park, viz.park_layer_style_mpl("park_boundary"), "NPS park boundary"),
        (glkn_park, viz.park_layer_style_mpl("reference"), "GLKN confirmed disturbance polygons"),
    ], title=f"{park} -- {GLKN_PARK_NAMES[park]}", ax=ax)
axes.flat[-1].axis("off")
fig.suptitle("GLKN confirmed disturbance polygons vs. NPS park boundary, all 7 parks", y=1.01, fontsize=13)
fig.tight_layout()


## 6. Key QA Questions (explicit)

Full numbers: `outputs/qa/glkn_processing_report.md` §Step 6.

In [ ]:
# Q1: multiple agents populated?
n_agents = glkn[["agent_01", "agent_02", "agent_03"]].notna().sum(axis=1)
print("Number of agents populated per confirmed polygon:")
print(n_agents.value_counts().sort_index())
print(f"\n{(n_agents>1).sum()} of {len(glkn)} confirmed polygons ({100*(n_agents>1).sum()/len(glkn):.2f}%) have more than one agent.")


In [ ]:
# Q2: uniqID repeat pattern -- polygon count vs. distinct physical patch count.
# NOTE: polygon counts are NOT called "event counts" or "patch counts" here or
# anywhere else in this notebook, per the source documentation's own caveat
# (individual disturbance events can span multiple polygons) and this repeat pattern.
uid_counts = glkn["uniqID"].value_counts()
print(f"Distinct uniqID values: {glkn['uniqID'].nunique():,} across {len(glkn):,} confirmed polygon rows")
print(f"-> raw polygon counts overstate distinct uniqID groups by "
      f"{100*(len(glkn)-glkn['uniqID'].nunique())/glkn['uniqID'].nunique():.1f}%")
print("\nRepeat-count distribution (how many uniqIDs appear 1x, 2x, 3x...):")
print(uid_counts.value_counts().sort_index())


In [ ]:
# Q3: confirmed disturbance area and counts by year and park (summary table; full detail is
# the year-histogram grid in Section 2)
by_year_park = glkn.groupby(["park_code", "year"]).agg(
    polygon_count=("source_feature_id", "count"),
    area_ha=("geometry", lambda g: g.area.sum() / 1e4),
).reset_index()
print(f"{len(by_year_park)} park-year combinations. Example (SLBE):")
by_year_park[by_year_park.park_code == "SLBE"].head(10)


In [ ]:
# Q4: obvious duplicate geometries risking double-counting?
dup_wkb = glkn.geometry.apply(lambda g: g.wkb).duplicated(keep=False)
print(f"{dup_wkb.sum()} of {len(glkn)} confirmed polygons share an exact duplicate geometry -- negligible.")


**Q5 (Isle Royale custom HUC codes):** answered in Section 4 — effectively removes ISRO from any HUC10-level summary unless resolved separately; not a minor edge case.

**Q6 (MISS `false`-row HUC_12 anomaly):** checked directly against confirmed rows only — 0 of MISS's 4,623 confirmed rows have missing `HUC_12`. The anomaly was specific to MISS's rejected (`false`) rows, entirely outside this product's scope. Resolved, does not affect this product.

## 7. Current Interpretation / Open Questions

### Measured (this notebook / `src/process_glkn.py`)

- Re-verified the full GLKN schema fresh against the prior inventory — total rows, true/false counts, and the 7-park set all matched exactly, with one real bug caught along the way (`change_occurred` stored as string `"true"`/`"false"`, not boolean).
- Geometry repair via `make_valid()` produced negligible area change (0.0% across all 7 parks) on confirmed disturbances, with invalidity rates (0.27%–4.43%) far lower than NCCN's.
- 35.78% of confirmed disturbance area cannot be assigned to a standard-derived HUC10, concentrated almost entirely in ISRO (99.9% of its confirmed rows).
- 1.43% of confirmed polygons have more than one agent; `uniqID` repeats such that polygon counts overstate distinct patch groups by roughly 15–16%.

### Documented (GLKN metadata + SLBE report)

- Disturbance polygons are human-interpreted LandTrendr candidates, not a wall-to-wall land-cover map (same distinction established for NCCN).
- False-positive (rejected) candidates are explicitly excluded from GLKN's own published summary analysis — consistent with this notebook's choice to exclude them from the primary product.
- GLKN's own SLBE report aggregates `HUC_12` to HUC10 for presentation, without stating that HUC10 is the underlying monitoring-extent boundary (the same caution applied to NCCN's HUC10/HUC12 test applies here — not yet tested for GLKN).

### 7.1 Isle Royale / Voyageurs non-standard HUC codes -- investigated, then paused

Checked GLKN_metadata.rtf, the Kirschbaum SLBE report, and the GDB's SCH_DATASET/SCH_RELEASE/SCH_UNIQUEID tables (confirmed these are pure ESRI internal schema/version-tracking tables, empty of any code definitions) -- none explain the non-standard codes.

**One clean, decisive finding from existing project data (not spatial inference):** every non-standard-code confirmed row, in both ISRO (16,023/16,023) and VOYA (926/926), has `loc_02=='canada'` -- a 100% correlation, zero exceptions either direction; all standard-code rows are `loc_02=='usa'`. `owner_type1` on the non-standard rows is dominated by "Crown Land Unpatented/Patent" (Canadian land-tenure terminology).

**This establishes *why* no standard code exists** -- the USGS Watershed Boundary Dataset (source of the standard 12-digit codes) does not cover Canadian territory -- **but not what the non-standard codes themselves specifically denote.** Investigation paused here per instruction, before completing spatial-coherence analysis or sourcing a Canadian boundary/code dataset.

**FINAL ANALYSIS BOUNDARY PENDING: standard US HUC boundaries alone are insufficient for ISRO/VOYA, because their attributed data extend into Canada**, where no boundary geometry (US or Canadian) currently exists in this project. This does not block the broader reference-data assessment -- the purpose of an analysis boundary here is only to provide a sensible landscape container for characterizing the attributed labels, not to reproduce GLKN's original monitoring geography.

### Interpretation / open questions (not yet settled)

- **Whether formal NPS park boundaries meaningfully under-represent GLKN's monitored extent**, the way they do for 3 of NCCN's 4 parks, has not yet been tested quantitatively (no inside/outside-boundary analysis was computed this pass). The maps in Section 5 make the *qualitative* pattern visible.
- Whether GLKN used a documented, reproducible "monitoring landscape" concept analogous to NCCN's "Protected Areas" study area has not been investigated in this pass.

### Next steps (not started)

- Quantify GLKN reference-polygon area inside vs. outside the formal park boundary, per park (NCCN Step 5 equivalent) -- deferred.
- Per-park dissolved HUC-based analysis boundaries for GLKN -- deferred (blocked on GLKN-region HUC boundary geometry, which does not yet exist in this project; ISRO/VOYA additionally need Canadian-side coverage, per 7.1).
- Move to ADS R6.

---
# ADS Region 6

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**Source:** the national **Insect & Disease Survey (IDS) database**, maintained by the USDA Forest Service's Forest Health Assessment and Applied Sciences Team (FHAAST) as part of the Forest Health Protection (FHP) program. Authorized by the Cooperative Forestry Assistance Act of 1978, Section 8 [16 U.S.C. 2104], directing the Forest Service to "conduct surveys to detect and appraise insect infestations and disease conditions and man-made stresses affecting trees... and report annually." (USFS *GIS Handbook and Data Conformity Standards*, October 2025, `docs/source_docs/ads/GIS-Handbook-for-Forest-Health-Detection-Survey.pdf`.)

**How it's produced:** the primary collection method is the **aerial detection survey** -- trained observers sketch-map tree damage in real time from an aircraft, using the Digital Mobile Sketch Mapping (DMSM) tablet system (production use since 2016). Ground survey and a developing remote-sensing component also contribute. Surveyors record each observation as a point, polygon, or grid cell, chosen based on how clearly the damage boundary can be discerned from the air -- the Handbook documents this explicitly as a source of surveyor-to-surveyor variability (a "lumper" vs. "splitter" mapping style can produce very different mapped footprints for the same underlying damage). Data pass through post-survey QA/QC before being finalized and compiled into the national IDS database.

**What this dataset explicitly is not:** per the source agency, "Detection surveys do not provide a full inventory of tree damage, but rather are an efficient and economical method of collecting and reporting out on the presence, extent, and severity of forest disturbances." It is a **repeated annual survey**, not a one-time census -- the same ground can be legitimately re-attributed with damage in different years.

**This section's data:** USFS **Region 6 (Pacific Northwest, primarily OR/WA)**, from `ADS_R6_Damage_allyears.shp`, a shapefile export of the same national IDS database (field names truncated to 10 characters, a shapefile fingerprint). Unlike the R10 File Geodatabase (next section), no separate damage-points or surveyed-extent layer was provided for R6 -- only this polygon export.

**Years:** the national IDS archive extends back to **1997**; this extract spans **1997-2025 (29 distinct years)**.

**What one record represents:** a single surveyor observation of tree damage at a location in a given survey year -- one specific host / damage-causal-agent / damage-type combination, at the surveyor's chosen feature type and drawn footprint. It is **not** a 1:1 stand-in for a discrete real-world disturbance event, and it is **not evidence of unique land area disturbed when summed across years** -- cumulative record/area totals are a repeated-survey sum, not a one-time census. Multiple observations legitimately sharing the same or overlapping footprint ("pancaked" features) are expected and documented by the source agency, not duplicates or errors: 0 of this section's 911,911 records are flagged `OBSERVATION_COUNT=='MULTIPLE'`.

In [ ]:
ads_r6 = gpd.read_parquet(DATA_PROCESSED / "ads_r6" / "ads_r6_region6_with_ecoregion.parquet")
ecoregions = gpd.read_parquet(DATA_PROCESSED / "boundaries" / "ads_r6_ecoregions.parquet")

print(f"Loaded {len(ads_r6):,} ADS R6 (REGION_ID==6) damage polygons, CRS={ads_r6.crs.to_epsg()}")
print(f"Loaded {len(ecoregions)} dissolved EPA Level III ecoregions")
print(f"Years represented: {int(ads_r6['SURVEY_YEA'].min())}-{int(ads_r6['SURVEY_YEA'].max())} "
      f"({ads_r6['SURVEY_YEA'].nunique()} distinct years)")


**Region filter QA** (re-verified, not assumed): the raw ADS R6 file contains a small number of stray non-Region-6 records despite the filename.

In [ ]:
region_qa = pd.read_csv(QA_DIR / "ads_r6_region_filter_qa.csv")
region_qa


### A.2 Basic dataset summary (this section, before any regional split)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(ads_r6),
    "attributed_area_ha": ads_r6["area_m2"].sum() / 1e4,
    "first_year": int(ads_r6["SURVEY_YEA"].min()),
    "last_year": int(ads_r6["SURVEY_YEA"].max()),
    "years_represented": int(ads_r6["SURVEY_YEA"].nunique()),
    "n_DCA_classes": int(ads_r6["DCA_CODE"].nunique()),
    "n_DAMAGE_TYPE_classes": int(ads_r6["DAMAGE_T_1"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `SURVEY_YEAR` | Year the survey was conducted |
| `REGION_ID` | USFS Region identifier (6 = Region 6 (Pacific Northwest, primarily OR/WA); 1,254 stray non-R6 records (`REGION_ID` 1 and 5) filtered out -- verified again rather than assumed, see below) |
| `DCA_CODE` / `DCA_COMMON` | **Damage Causal Agent** -- the specific insect, disease, or abiotic agent responsible (national code list, 1,000+ possible codes; this section uses 91 distinct codes) |
| `DAMAGE_TYP` / `DAMAGE_T_1` | Type of damage observed (e.g., mortality, discoloration, defoliation) -- a second, coarser native taxonomy dimension alongside DCA |
| `PERCENT_AFFECTED_CODE` / `PERCENT_AFFECTED` / `PERCENT_MIN`/`MAX`/`MID` | **Damage intensity**: % of standing (live + dead) trees *within the polygon* affected, on a 5-class scale (Very Light 1-3% to Very Severe >50%) -- not the % of the polygon's total area |
| `HOST` / `HOST_GROUP` | Tree species or host group affected |
| `OBSERVATION_COUNT` / `OBSERVAT_1` | Flags whether a footprint has multiple co-located ("pancaked") observations |
| `AREA_TYPE` | Feature type recorded (POLYGON here) |
| `COLLECTION_MODE` | Aerial_Survey / Ground_Check / Ground_Survey / Scan_Sketch / DesktopGIS -- how the observation was collected |
| `OBJECTID` | Row/footprint identifier |
| `STATUS` | Data-finality flag in this extract |


### A.4 Overall native attribution distribution (DCA and DAMAGE_TYPE, all regions combined)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

dca_area = ads_r6.groupby("DCA_COMMON")["area_m2"].sum().sort_values(ascending=False).head(15) / 1e4
ax1.barh(dca_area.index[::-1], dca_area.values[::-1], color="#4575b4")
ax1.set_xlabel("attributed area (ha)")
ax1.set_title(f"ADS R6 top 15 DCA (of {ads_r6['DCA_CODE'].nunique()} total), all regions combined")

dmg_area = ads_r6.groupby("DAMAGE_T_1")["area_m2"].sum().sort_values(ascending=False) / 1e4
ax2.barh(dmg_area.index[::-1], dmg_area.values[::-1], color="#d73027")
ax2.set_xlabel("attributed area (ha)")
ax2.set_title(f"ADS R6 DAMAGE_T_1 ({ads_r6['DAMAGE_T_1'].nunique()} total), all regions combined")

fig.tight_layout()


### A.5 Overall spatial distribution (full dataset)

A 2D density view of ADS R6 polygon centroids handles all ~912,000 points natively (unlike plotting every polygon outline) and directly answers "where are the concentrations" -- the actual question motivating this analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ecoregions.boundary.plot(ax=ax, color="black", linewidth=1)
centroids = ads_r6.geometry.centroid
hb = ax.hexbin(centroids.x, centroids.y, gridsize=80, cmap="inferno", bins="log", mincnt=1)
fig.colorbar(hb, ax=ax, label="log10(polygon count) per hex cell")
ax.set_title(f"ADS R6 damage polygon density (n={len(ads_r6):,}), EPA ecoregion boundaries for context")
ax.set_aspect("equal")
fig.tight_layout()


### A.6 Overall temporal distribution (all regions combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_year.csv")
year_area = by_year.groupby("SURVEY_YEA")["area_m2"].sum() / 1e4  # ha, all ecoregions combined

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("attributed area (ha)")
ax.set_xlabel("survey year")
ax.set_title(f"ADS R6 attributed area by year, all ecoregions combined "
             f"({int(by_year.SURVEY_YEA.min())}-{int(by_year.SURVEY_YEA.max())}, "
             f"{by_year.SURVEY_YEA.nunique()} distinct years)")
fig.tight_layout()


### A.7 Illustrative polygon sample (labeled, for shape inspection)

A random sample of individual polygons -- **not the full dataset** -- shown against ecoregion boundaries, so individual attributed shapes can be visually inspected. The density map above (A.5) is the representative view of the full data. *(Informal/illustrative QA visualization -- candidate for an appendix in the final report.)*

In [ ]:
SAMPLE_N = 3000
sample = ads_r6.sample(n=min(SAMPLE_N, len(ads_r6)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 10))
viz.static_overlay_map([
    (ecoregions, dict(facecolor="none", edgecolor="black", linewidth=1.2), "EPA ecoregions"),
    (sample, dict(facecolor="#d73027", edgecolor="none", alpha=0.6), f"ADS R6 sample (n={len(sample):,} of {len(ads_r6):,})"),
], title="ADS R6 -- random polygon sample against ecoregion boundaries", ax=ax)
fig.tight_layout()


In [ ]:
# Interactive equivalent -- same sample (the full 913k-polygon dataset is not
# feasible to embed in an interactive map).
m = viz.interactive_overlay_map([
    (ecoregions, dict(color="black", weight=1.5, fillOpacity=0.0), "EPA ecoregions", True),
    (sample, viz.park_layer_style("reference"), f"ADS R6 sample (n={len(sample):,})", True),
], zoom_start=6)
m


---
## PART B --- Regional / Zonal Assessment

Having characterized ADS R6 as a whole (Part A), we now divide its landscape into the 7 dissolved EPA Level III ecoregions used as candidate analysis regions and examine how attributed records, area, and native attribution classes are distributed -- and change through time -- within each. Region boundaries are used directly as delivered/derived; see Part A and `outputs/qa/ads_r6_processing_report.md` for how they were constructed.

### B.1 Region definitions: EPA Level III ecoregions (dissolved)

In [ ]:
ecoregions[["us_l3code", "us_l3name", "n_parts", "region_area_m2"]].assign(
    region_area_km2=lambda d: d["region_area_m2"] / 1e6
)[["us_l3code", "us_l3name", "n_parts", "region_area_km2"]]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ecoregions.plot(ax=ax, column="us_l3name", cmap="tab10", alpha=0.5, edgecolor="black", linewidth=0.8, legend=True,
                 legend_kwds={"loc": "lower left", "fontsize": 7})
ax.set_title("BugNet R6 EPA Level III ecoregions, dissolved (7 regions)")
ax.set_aspect("equal")
fig.tight_layout()


### B.2 Attributed records and area by region

**Primary metrics: polygon/record count and attributed area** -- not the cumulative-area-to-region-area ratio (see the QA/context note at the end of this subsection for why).

In [ ]:
capture = pd.read_csv(QA_DIR / "ads_r6_ecoregion_capture.csv")
capture[["us_l3code", "us_l3name", "polygon_count", "attributed_area_m2"]].assign(
    attributed_area_ha=lambda d: d["attributed_area_m2"] / 1e4
)[["us_l3code", "us_l3name", "polygon_count", "attributed_area_ha"]]


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
plot_df = capture.dropna(subset=["us_l3code"]).sort_values("attributed_area_m2", ascending=False)

ax1.bar(plot_df["us_l3name"], plot_df["polygon_count"], color="#4575b4")
ax1.set_ylabel("attributed polygon count")
ax1.set_title("ADS R6 attributed record count by ecoregion")
ax1.tick_params(axis="x", rotation=30)
for tick in ax1.get_xticklabels():
    tick.set_ha("right")

ax2.bar(plot_df["us_l3name"], plot_df["attributed_area_m2"] / 1e4, color="#4575b4")
ax2.set_ylabel("attributed area (ha)")
ax2.set_title("ADS R6 attributed area by ecoregion")
ax2.tick_params(axis="x", rotation=30)
for tick in ax2.get_xticklabels():
    tick.set_ha("right")

fig.tight_layout()


**Total: 911,911 attributed records, 20,301,637 ha, across 7 ecoregions.** 97.0% of total attributed area falls inside the union of the 7 ecoregions (6.07×10⁹ m² / 3.0% falls outside — not yet investigated further, per instruction not to spend more time optimizing capture). 70,464 polygons have >50% of their own area outside all 7 ecoregions.

**QA/context note — cumulative "fill fraction" (not a primary metric here):** dividing attributed area by each ecoregion's static land area gives a number that can legitimately exceed 100% for a multi-year dataset like this one, because the same ground can be validly re-attributed with damage in different years. `SURVEY_YEA` spans 1997–2025 (up to 29 distinct years in one ecoregion), and two ecoregions (Eastern Cascades Slopes and Foothills, North Cascades) do exceed 100% cumulative fill — verified as real multi-year richness, not double-counting (the `DAMAGE_ARE` event-ID repeat rate there is only ~1.06×, not the driver). Kept as QA/context, not emphasized as a primary result. Full detail: `outputs/qa/ads_r6_processing_report.md`.

### B.3 Region through time

In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_year.csv")
year_pivot = by_year.pivot(index="SURVEY_YEA", columns="us_l3code", values="polygon_count").fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))
year_pivot.plot(ax=ax, linewidth=1.2)
ax.set_ylabel("polygon count")
ax.set_xlabel("survey year")
ax.set_title("ADS R6 attributed record count by year, per ecoregion (see the heatmap below for area instead of count)")
ax.legend(title="us_l3code", fontsize=7, ncol=2)
fig.tight_layout()


In [ ]:
year_area_region = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_year.csv")
year_area_region = year_area_region.merge(
    capture[["us_l3code", "us_l3name"]].dropna(), on="us_l3code", how="left"
)
heat = year_area_region.pivot_table(index="us_l3name", columns="SURVEY_YEA", values="area_m2", aggfunc="sum").fillna(0) / 1e4
region_order = capture.dropna(subset=["us_l3code"]).sort_values("attributed_area_m2", ascending=False)["us_l3name"]
heat = heat.reindex(region_order)

fig, ax = plt.subplots(figsize=(14, 0.4 * len(heat) + 2))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=8)
ax.set_xlabel("survey year")
ax.set_title("ADS R6 attributed area (ha) by region x year")
fig.colorbar(im, ax=ax, label="attributed area (ha)")
fig.tight_layout()


### B.4 Class composition by region: DCA_CODE and DAMAGE_TYP

Native ADS taxonomy preserved verbatim -- not harmonized with NCCN/GLKN/ADS R10.

In [ ]:
by_dca = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_dca.csv")
print("Top 5 causal agents (DCA_COMMON) by area, per ecoregion:")
for code_, grp in by_dca.groupby("us_l3code"):
    top5 = grp.sort_values("area_m2", ascending=False).head(5)
    print(f"\n{code_}:")
    for r in top5.itertuples():
        print(f"  {r.DCA_COMMON}: {r.polygon_count:,} polygons, {r.area_m2/1e4:,.0f} ha")


In [ ]:
by_damage_typ = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_damage_type.csv")
damage_pivot = by_damage_typ.groupby(["us_l3code", "DAMAGE_T_1"])["area_m2"].sum().unstack(fill_value=0)
damage_pivot_pct = damage_pivot.div(damage_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
damage_pivot_pct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("% of ecoregion's attributed area")
ax.set_title("ADS R6 damage type (DAMAGE_T_1) composition by ecoregion")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
fig.tight_layout()


### B.5 Class composition through time within region

In [ ]:
stack = pd.read_csv(QA_DIR / "ads_r6_by_ecoregion_year_dca.csv")

TOP_N = 8
top_classes = stack.groupby("DCA_COMMON")["area_m2"].sum().sort_values(ascending=False).head(TOP_N).index.tolist()
stack["dca_bucket"] = stack["DCA_COMMON"].where(stack["DCA_COMMON"].isin(top_classes), "Other")

region_order = capture.dropna(subset=["us_l3code"]).sort_values("attributed_area_m2", ascending=False)["us_l3name"].tolist()
region_code_by_name = capture.set_index("us_l3name")["us_l3code"].to_dict()

palette = plt.get_cmap("tab10").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

ncols = 4
nrows = -(-len(region_order) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.6 * nrows), sharex=False)
axes_flat = axes.flat if hasattr(axes, "flat") else [axes]
for ax, region_name in zip(axes_flat, region_order):
    code_ = region_code_by_name[region_name]
    sub = stack[stack["us_l3code"] == code_]
    pivot = sub.groupby(["SURVEY_YEA", "dca_bucket"])["area_m2"].sum().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot_pct.columns:
            continue
        vals = pivot_pct[cls].values
        ax.bar(pivot_pct.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(region_name, fontsize=8)
    ax.tick_params(labelsize=6)
for ax in list(axes_flat)[len(region_order):]:
    ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.03))
fig.suptitle("ADS R6: DCA composition (% of attributed area) by year, within each region (top 8 DCA + Other)", y=1.01)
fig.tight_layout()


## Interpretation / Summary

### Measured

- 911,911 attributed records (`REGION_ID==6`), 20,301,637 ha total attributed area, spanning 1997-2025 (29 distinct survey years).
- 97.0% of attributed area falls inside the union of the 7 dissolved ecoregions; 7.7% of polygons have a majority of their own area outside all 7.
- Damage concentration (A.5) and causal-agent/damage-type composition (B.4) both vary meaningfully across ecoregions and years.
- The region x year heatmap (B.3) and class-through-time small multiples (B.5) show this variation is not a smooth background rate -- individual ecoregions have concentrated multi-year runs for particular causal agents (e.g., mountain pine beetle outbreaks).
- Two data-quality issues caught and fixed during processing: a >12-minute naive geometric overlay (fixed with a prepared-geometry short-circuit) and a per-ecoregion area bug that produced impossible >100% cumulative fill fractions for two ecoregions (fixed; the >100% values that remain are verified as real multi-year richness, not the bug).

### Not yet done (by design, per instruction)

- No cross-source class harmonization (native `DCA_CODE`/`DAMAGE_TYP` preserved as-is).
- No attempt to reconstruct ADS's original historical analysis boundaries.
- No further optimization of the 97.0% ecoregion capture rate.
- No final reference-data summary statistics, no focal-area selection.

---
# ADS Region 10

## PART A --- Source Dataset Description / Characterization

### A.1 What is this dataset, who produced it, and how?

**Source:** the national **Insect & Disease Survey (IDS) database**, maintained by the USDA Forest Service's Forest Health Assessment and Applied Sciences Team (FHAAST) as part of the Forest Health Protection (FHP) program. Authorized by the Cooperative Forestry Assistance Act of 1978, Section 8 [16 U.S.C. 2104], directing the Forest Service to "conduct surveys to detect and appraise insect infestations and disease conditions and man-made stresses affecting trees... and report annually." (USFS *GIS Handbook and Data Conformity Standards*, October 2025, `docs/source_docs/ads/GIS-Handbook-for-Forest-Health-Detection-Survey.pdf`.)

**How it's produced:** the primary collection method is the **aerial detection survey** -- trained observers sketch-map tree damage in real time from an aircraft, using the Digital Mobile Sketch Mapping (DMSM) tablet system (production use since 2016). Ground survey and a developing remote-sensing component also contribute. Surveyors record each observation as a point, polygon, or grid cell, chosen based on how clearly the damage boundary can be discerned from the air -- the Handbook documents this explicitly as a source of surveyor-to-surveyor variability (a "lumper" vs. "splitter" mapping style can produce very different mapped footprints for the same underlying damage). Data pass through post-survey QA/QC before being finalized and compiled into the national IDS database.

**What this dataset explicitly is not:** per the source agency, "Detection surveys do not provide a full inventory of tree damage, but rather are an efficient and economical method of collecting and reporting out on the presence, extent, and severity of forest disturbances." It is a **repeated annual survey**, not a one-time census -- the same ground can be legitimately re-attributed with damage in different years.

**This section's data:** USFS **Region 10 (Alaska)**, from `DAMAGE_AREAS_FLAT_AllYears_AK_Rgn10`, a File Geodatabase layer (`AK_Region10_AllYears.gdb`, extracted from the raw ZIP to a persistent working copy; raw ZIP itself untouched). Field names are untruncated (unlike the ADS R6 shapefile export in the previous section), and this GDB also contains a damage-points layer and a dedicated surveyed-area-extent layer (not yet used in this assessment).

**Years:** the national IDS archive extends back to **1997**; this extract spans **1997-2025 (29 distinct years)**.

**What one record represents:** a single surveyor observation of tree damage at a location in a given survey year -- one specific host / damage-causal-agent / damage-type combination, at the surveyor's chosen feature type and drawn footprint. It is **not** a 1:1 stand-in for a discrete real-world disturbance event, and it is **not evidence of unique land area disturbed when summed across years** -- cumulative record/area totals are a repeated-survey sum, not a one-time census. Multiple observations legitimately sharing the same or overlapping footprint ("pancaked" features) are expected and documented by the source agency, not duplicates or errors: 6,949 of this section's 151,309 records are flagged `OBSERVATION_COUNT=='MULTIPLE'`.

In [ ]:
ads_r10 = gpd.read_parquet(DATA_PROCESSED / "ads_r10" / "ads_r10_with_huc6.parquet")
huc6 = gpd.read_parquet(DATA_PROCESSED / "boundaries" / "ads_r10_huc6.parquet")

print(f"Loaded {len(ads_r10):,} ADS R10 (REGION_ID==10) damage polygons, CRS={ads_r10.crs.to_epsg()}")
print(f"Loaded {len(huc6)} BugNet R10 HUC6 regions")
print(f"Years represented: {int(ads_r10['SURVEY_YEAR'].min())}-{int(ads_r10['SURVEY_YEAR'].max())} "
      f"({ads_r10['SURVEY_YEAR'].nunique()} distinct years)")
print(f"100% REGION_ID==10 -- no region filter needed (unlike R6, which had stray non-6 records).")

n_multi = int((ads_r10["OBSERVATION_COUNT"] == "MULTIPLE").sum())
n_distinct_footprint = ads_r10["DAMAGE_AREA_ID"].nunique()
print(f"\nPancake QA: {n_multi:,} rows flagged OBSERVATION_COUNT=='MULTIPLE'; "
      f"{n_distinct_footprint:,} distinct DAMAGE_AREA_ID footprints of {len(ads_r10):,} total rows "
      f"({len(ads_r10) - n_distinct_footprint:,} 'extra' rows -- legitimate distinct co-located "
      f"observations per the official USDA readme, not duplicates).")


### A.2 Basic dataset summary (this section, before any regional split)

In [ ]:
summary_row = pd.DataFrame([{
    "record_count": len(ads_r10),
    "attributed_area_ha": ads_r10["area_m2"].sum() / 1e4,
    "first_year": int(ads_r10["SURVEY_YEAR"].min()),
    "last_year": int(ads_r10["SURVEY_YEAR"].max()),
    "years_represented": int(ads_r10["SURVEY_YEAR"].nunique()),
    "n_DCA_classes": int(ads_r10["DCA_CODE"].nunique()),
    "n_DAMAGE_TYPE_classes": int(ads_r10["DAMAGE_TYPE"].nunique()),
}])
summary_row


### A.3 Attribute / schema table

Focused on fields relevant to interpreting and using the reference data -- not every GIS field.

| Field | Meaning |
|---|---|
| `SURVEY_YEAR` | Year the survey was conducted |
| `REGION_ID` | USFS Region identifier (10 = Region 10 (Alaska); verified 100% here, no filtering needed) |
| `DCA_CODE` / `DCA_COMMON_NAME` | **Damage Causal Agent** -- the specific insect, disease, or abiotic agent responsible (national code list, 1,000+ possible codes; this section uses 70 distinct codes) |
| `DAMAGE_TYPE_CODE` / `DAMAGE_TYPE` | Type of damage observed (e.g., mortality, discoloration, defoliation) -- a second, coarser native taxonomy dimension alongside DCA |
| `PERCENT_AFFECTED_CODE` / `PERCENT_AFFECTED` / `PERCENT_MIN`/`MAX`/`MID` | **Damage intensity**: % of standing (live + dead) trees *within the polygon* affected, on a 5-class scale (Very Light 1-3% to Very Severe >50%) -- not the % of the polygon's total area |
| `HOST` / `HOST_GROUP` | Tree species or host group affected |
| `OBSERVATION_COUNT` / `OBSERVATION_ID` | Flags whether a footprint has multiple co-located ("pancaked") observations |
| `AREA_TYPE` | Feature type recorded (POLYGON here) |
| `COLLECTION_MODE` | Aerial_Survey / Ground_Check / Ground_Survey / Scan_Sketch / DesktopGIS -- how the observation was collected |
| `DAMAGE_AREA_ID` | Row/footprint identifier |
| `STATUS` | Data-finality flag in this extract |


### A.4 Overall native attribution distribution (DCA and DAMAGE_TYPE, all regions combined)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

dca_area = ads_r10.groupby("DCA_COMMON_NAME")["area_m2"].sum().sort_values(ascending=False).head(15) / 1e4
ax1.barh(dca_area.index[::-1], dca_area.values[::-1], color="#4575b4")
ax1.set_xlabel("attributed area (ha)")
ax1.set_title(f"ADS R10 top 15 DCA (of {ads_r10['DCA_CODE'].nunique()} total), all regions combined")

dmg_area = ads_r10.groupby("DAMAGE_TYPE")["area_m2"].sum().sort_values(ascending=False) / 1e4
ax2.barh(dmg_area.index[::-1], dmg_area.values[::-1], color="#d73027")
ax2.set_xlabel("attributed area (ha)")
ax2.set_title(f"ADS R10 DAMAGE_TYPE ({ads_r10['DAMAGE_TYPE'].nunique()} total), all regions combined")

fig.tight_layout()


### A.5 Overall spatial distribution (full dataset)

A 2D density view of ADS R10 polygon centroids handles all ~151,000 points natively -- the actual question motivating this analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
huc6.boundary.plot(ax=ax, color="black", linewidth=1)
centroids = ads_r10.geometry.centroid
hb = ax.hexbin(centroids.x, centroids.y, gridsize=80, cmap="inferno", bins="log", mincnt=1)
fig.colorbar(hb, ax=ax, label="log10(polygon count) per hex cell")
ax.set_title(f"ADS R10 damage polygon density (n={len(ads_r10):,}), HUC6 boundaries for context")
ax.set_aspect("equal")
fig.tight_layout()


### A.6 Overall temporal distribution (all regions combined)

In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r10_by_huc6_year.csv")
year_area = by_year.groupby("SURVEY_YEAR")["area_m2"].sum() / 1e4  # ha, all HUC6 combined

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(year_area.index, year_area.values, color="#4575b4")
ax.set_ylabel("attributed area (ha)")
ax.set_xlabel("survey year")
ax.set_title(f"ADS R10 attributed area by year, all HUC6 regions combined "
             f"({int(by_year.SURVEY_YEAR.min())}-{int(by_year.SURVEY_YEAR.max())}, "
             f"{by_year.SURVEY_YEAR.nunique()} distinct years)")
fig.tight_layout()


### A.7 Illustrative polygon sample (labeled, for shape inspection)

A random sample of individual polygons -- **not the full dataset** -- shown against HUC6 boundaries, so individual attributed shapes can be visually inspected. The density map above (A.5) is the representative view of the full data. *(Informal/illustrative QA visualization -- candidate for an appendix in the final report.)*

In [ ]:
SAMPLE_N = 3000
sample = ads_r10.sample(n=min(SAMPLE_N, len(ads_r10)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 10))
viz.static_overlay_map([
    (huc6, dict(facecolor="none", edgecolor="black", linewidth=1.2), "HUC6 regions"),
    (sample, dict(facecolor="#d73027", edgecolor="none", alpha=0.6), f"ADS R10 sample (n={len(sample):,} of {len(ads_r10):,})"),
], title="ADS R10 -- random polygon sample against HUC6 boundaries", ax=ax)
fig.tight_layout()


In [ ]:
# Interactive equivalent -- same sample (the full 151k-polygon dataset is not
# feasible to embed in an interactive map).
m = viz.interactive_overlay_map([
    (huc6, dict(color="black", weight=1.5, fillOpacity=0.0), "HUC6 regions", True),
    (sample, viz.park_layer_style("reference"), f"ADS R10 sample (n={len(sample):,})", True),
], zoom_start=5)
m


---
## PART B --- Regional / Zonal Assessment

Having characterized ADS R10 as a whole (Part A), we now divide its landscape into the 20 HUC6 watershed basins used as candidate analysis regions and examine how attributed records, area, and native attribution classes are distributed -- and change through time -- within each. Region boundaries are used directly as delivered/derived; see Part A and `outputs/qa/ads_r10_processing_report.md` for how they were constructed.

### B.1 Region definitions: HUC6 boundaries (no dissolve needed)

In [ ]:
huc6[["huc6", "name", "states", "n_parts"]].assign(
    region_area_km2=lambda d: huc6["region_area_m2"] / 1e6
)[["huc6", "name", "states", "n_parts", "region_area_km2"]]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
huc6.plot(ax=ax, column="name", cmap="tab20", alpha=0.5, edgecolor="black", linewidth=0.8, legend=True,
          legend_kwds={"loc": "lower left", "fontsize": 6, "ncol": 2})
ax.set_title("BugNet R10 HUC6 regions, Alaska (20 regions, no dissolve needed)")
ax.set_aspect("equal")
fig.tight_layout()


### B.2 Attributed records and area by region

**Primary metrics: polygon/record count and attributed area** -- not the cumulative-area-to-region-area ratio (see the QA/context note below).

In [ ]:
capture = pd.read_csv(QA_DIR / "ads_r10_huc6_capture.csv")
capture[["huc6_code", "huc6_name", "record_count", "attributed_area_m2"]].assign(
    attributed_area_ha=lambda d: d["attributed_area_m2"] / 1e4
)[["huc6_code", "huc6_name", "record_count", "attributed_area_ha"]]


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
plot_df = capture.dropna(subset=["huc6_code"]).sort_values("attributed_area_m2", ascending=False)

ax1.bar(plot_df["huc6_name"], plot_df["record_count"], color="#4575b4")
ax1.set_ylabel("attributed record count")
ax1.set_title("ADS R10 attributed record count by HUC6")
ax1.tick_params(axis="x", rotation=75)
for tick in ax1.get_xticklabels():
    tick.set_ha("right")
    tick.set_fontsize(7)

ax2.bar(plot_df["huc6_name"], plot_df["attributed_area_m2"] / 1e4, color="#d73027")
ax2.set_ylabel("attributed area (ha)")
ax2.set_title("ADS R10 attributed area by HUC6")
ax2.tick_params(axis="x", rotation=75)
for tick in ax2.get_xticklabels():
    tick.set_ha("right")
    tick.set_fontsize(7)

fig.tight_layout()


**Total: 151,309 attributed records, 9,825,155 ha, across 20 HUC6 regions.** 90.5% of total attributed area falls inside the union of the 20 HUC6 regions (9.29×10⁹ m² / 9.5% falls outside — lower capture than R6's 97.0%, reported as-is per instruction, not investigated further). No HUC6 exceeds 100% fill fraction (max: Kenai Peninsula, 31.4%) — unlike two R6 ecoregions.

**QA/context note — cumulative "fill fraction" (not a primary metric here):** dividing attributed area by each HUC6's static land area can, in principle, exceed 100% for the same reason documented for R6 — ADS spans many years (1997-2025 here too) and the same ground can be legitimately re-attributed with damage across different years. It does not happen for any HUC6 in R10, but the ratio is still not used as a primary metric, for consistency with R6 and because it remains sensitive to region size choice rather than data richness.

### B.3 Region through time

In [ ]:
by_year = pd.read_csv(QA_DIR / "ads_r10_by_huc6_year.csv")
year_pivot = by_year.pivot(index="SURVEY_YEAR", columns="huc6_code", values="record_count").fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))
year_pivot.plot(ax=ax, linewidth=1.2, legend=False)
ax.set_ylabel("record count")
ax.set_xlabel("survey year")
ax.set_title("ADS R10 attributed record count by year, per HUC6 (legend omitted -- 20 series; see the heatmap below for a legible per-region view)")
fig.tight_layout()


In [ ]:
year_area_region = pd.read_csv(QA_DIR / "ads_r10_by_huc6_year.csv")
year_area_region = year_area_region.merge(
    capture[["huc6_code", "huc6_name"]].dropna(), on="huc6_code", how="left"
)
heat = year_area_region.pivot_table(index="huc6_name", columns="SURVEY_YEAR", values="area_m2", aggfunc="sum").fillna(0) / 1e4
region_order = capture.dropna(subset=["huc6_code"]).sort_values("attributed_area_m2", ascending=False)["huc6_name"]
heat = heat.reindex(region_order)

fig, ax = plt.subplots(figsize=(14, 0.4 * len(heat) + 2))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=8)
ax.set_xlabel("survey year")
ax.set_title("ADS R10 attributed area (ha) by region x year")
fig.colorbar(im, ax=ax, label="attributed area (ha)")
fig.tight_layout()


### B.4 Class composition by region: DCA_CODE and DAMAGE_TYPE

Native ADS taxonomy preserved verbatim -- not harmonized with NCCN/GLKN/ADS R6.

In [ ]:
by_dca = pd.read_csv(QA_DIR / "ads_r10_by_huc6_dca.csv")
print("Top 5 causal agents (DCA_COMMON_NAME) by area, per HUC6:")
for code_, grp in by_dca.groupby("huc6_code"):
    top5 = grp.sort_values("area_m2", ascending=False).head(5)
    print(f"\n{code_}:")
    for r in top5.itertuples():
        print(f"  {r.DCA_COMMON_NAME}: {r.record_count:,} records, {r.area_m2/1e4:,.0f} ha")


In [ ]:
by_damage_typ = pd.read_csv(QA_DIR / "ads_r10_by_huc6_damage_type.csv")
damage_pivot = by_damage_typ.groupby(["huc6_code", "DAMAGE_TYPE"])["area_m2"].sum().unstack(fill_value=0)
damage_pivot_pct = damage_pivot.div(damage_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 6))
damage_pivot_pct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("% of HUC6's attributed area")
ax.set_title("ADS R10 damage type composition by HUC6")
ax.legend(title="DAMAGE_TYPE", fontsize=6, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=75)
fig.tight_layout()


### B.5 Class composition through time within region

In [ ]:
stack = pd.read_csv(QA_DIR / "ads_r10_by_huc6_year_dca.csv")

TOP_N = 8
top_classes = stack.groupby("DCA_COMMON_NAME")["area_m2"].sum().sort_values(ascending=False).head(TOP_N).index.tolist()
stack["dca_bucket"] = stack["DCA_COMMON_NAME"].where(stack["DCA_COMMON_NAME"].isin(top_classes), "Other")

region_order = capture.dropna(subset=["huc6_code"]).sort_values("attributed_area_m2", ascending=False)["huc6_name"].tolist()
region_code_by_name = capture.set_index("huc6_name")["huc6_code"].to_dict()

palette = plt.get_cmap("tab10").colors
color_map = {cls: palette[i % len(palette)] for i, cls in enumerate(top_classes)}
color_map["Other"] = "#999999"

ncols = 4
nrows = -(-len(region_order) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.6 * nrows), sharex=False)
axes_flat = axes.flat if hasattr(axes, "flat") else [axes]
for ax, region_name in zip(axes_flat, region_order):
    code_ = region_code_by_name[region_name]
    sub = stack[stack["huc6_code"] == code_]
    pivot = sub.groupby(["SURVEY_YEAR", "dca_bucket"])["area_m2"].sum().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    bottom = None
    for cls in list(top_classes) + ["Other"]:
        if cls not in pivot_pct.columns:
            continue
        vals = pivot_pct[cls].values
        ax.bar(pivot_pct.index, vals, bottom=bottom, color=color_map[cls], width=1.0)
        bottom = vals if bottom is None else bottom + vals
    ax.set_title(region_name, fontsize=8)
    ax.tick_params(labelsize=6)
for ax in list(axes_flat)[len(region_order):]:
    ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in list(top_classes) + ["Other"]]
fig.legend(handles, list(top_classes) + ["Other"], loc="lower center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.03))
fig.suptitle("ADS R10: DCA composition (% of attributed area) by year, within each region (top 8 DCA + Other)", y=1.01)
fig.tight_layout()


## Interpretation / Summary

### Measured

- 151,309 attributed records (`REGION_ID==10`, no filtering needed), 9,825,155 ha total attributed area, spanning 1997-2025 (29 distinct survey years) -- matches R6's span exactly.
- 90.5% of attributed area falls inside the union of the 20 HUC6 regions (lower than R6's 97.0%; ~7,300 records, 9.5% of area, fall outside all HUC6s -- reported as-is, not investigated further).
- No HUC6 exceeds 100% fill fraction (max: Kenai Peninsula, 31.4%), unlike two R6 ecoregions -- consistent with HUC6 basins being larger watershed-scale containers relative to attributed-damage density here.
- Damage concentration (A.5) and causal-agent/damage-type composition (B.4) both vary strongly by HUC6, with a clear geographic split: bark beetles (spruce beetle, etc.) dominate the southern/coastal HUC6s (Kenai Peninsula, Susitna River, Copper River), while defoliators/leafminers (aspen leafminer, willow leaf blotchminer, birch leafroller) dominate the northern/interior HUC6s (Tanana River, Porcupine River, Beaver Creek).
- The region x year heatmap (B.3) and class-through-time small multiples (B.5) show this composition is not static -- individual HUC6s have years of concentrated activity for a given DCA rather than a uniform background rate.
- One data-quality issue caught and fixed during processing: this GDB export has no `OBJECTID` column (unlike the readme's generic field description implied); `DAMAGE_AREA_ID` used as the row identifier throughout instead.

### Not yet done (by design, per instruction)

- No cross-source class harmonization (native `DCA_CODE`/`DAMAGE_TYPE` preserved as-is).
- No attempt to reconstruct ADS's original historical analysis boundaries.
- No investigation of the 9.5% uncaptured area or the Alaska/Canada-spanning HUC6s.
- No final reference-data summary statistics, no focal-area selection.

---
# Reference Data Assessment --- Current Status

This section summarizes what has been characterized across the four sources prepared so far. It is **not** a cross-source comparison, ranking, or harmonization -- each source above was assessed within its own geographic framework, native taxonomy, and methodology, for the reasons documented in each Part A section.

## What has been characterized, per source

| Source | Dataset description (Part A) | Regional assessment (Part B) | Candidate analysis regions |
|---|---|---|---|
| NCCN | Done -- provenance, method (LandTrendr), schema, native `change_class` distribution, temporal range, spatial footprint | Done -- regional summary, region x year, class composition by region and through time | 4 parks (own attributed-data footprint; no agreed containing boundary for 3 of 4 -- see below) |
| GLKN | Done -- provenance, method (LandTrendr + manual air-photo review), schema, native `agent_01` distribution, temporal range, spatial footprint | Done -- regional summary, region x year, class composition by region and through time | 7 parks (own confirmed-disturbance footprint) |
| ADS R6 | Done -- provenance (USDA FHAAST/IDS), method (aerial detection survey/DMSM), schema, native DCA/DAMAGE_TYPE distribution, temporal range, spatial footprint | Done -- regional summary, region x year, class composition by region and through time | 7 dissolved EPA Level III ecoregions |
| ADS R10 | Done -- same as R6, Alaska-specific (full GDB, untruncated fields) | Done -- regional summary, region x year, class composition by region and through time | 20 HUC6 watershed basins |

## Remaining data/boundary limitations (per source, not resolved here)

- **NCCN:** no agreed analysis-region boundary for MORA, NOCA, or LEWI -- the NPS park boundary excludes 89.6-99.1% of attributed area for those 3 parks, and neither HUC10 nor HUC12 shows meaningful edge alignment with the reference-data footprint. Blocked on the original "Protected Areas" study-area boundary (waiting on Natasha).
- **GLKN:** Isle Royale (99.9% of its own confirmed rows) and part of Voyageurs have no usable HUC/boundary geometry -- their attributed data extend into Canada, where no standard HUC code or boundary geometry exists in this project. No inside/outside-park-boundary analysis has been run for GLKN (unlike NCCN).
- **ADS R6:** 3.0% of attributed area falls outside the 7-ecoregion union; not investigated further (accepted as-is per instruction).
- **ADS R10:** 9.5% of attributed area falls outside the 20-HUC6 union -- larger than R6's uncaptured share; not investigated further. Several HUC6 basins extend into Canada.
- **Cross-cutting:** native class taxonomies (NCCN `change_class`, GLKN `agent_01`, ADS `DCA_CODE`/`DAMAGE_TYPE`) remain unharmonized by design -- they describe different constructs (change type vs. causal agent) and were not intended to be directly comparable at this stage.

## What was added in this reporting pass

- Region x year x native-class ("stack") tables for all four sources, built from already-processed data: `nccn_by_park_year_class.csv`, `glkn_by_park_year_class.csv`, `ads_r6_by_ecoregion_year_dca.csv`, `ads_r10_by_huc6_year_dca.csv`.
- Per-source Part A (dataset description) and Part B (regional/zonal assessment) structure, each with a region x year heatmap and a class-composition-through-time small-multiples figure.
- The USDA Forest Service *GIS Handbook and Data Conformity Standards* (Oct. 2025) was added to `docs/source_docs/ads/` and used, alongside the existing `IDS_FlatFiles_Readme.pdf`, to document ADS's provenance, collection method, and schema from official source material rather than general knowledge.

## Not done, by design

- No focal-area selection. No criteria-based ranking of regions.
- No cross-source class harmonization.
- No further boundary investigation beyond what each source's Part A/B already documents.
- Technical processing QA (geometry repair detail, HUC10/HUC12 fit testing, boundary-relationship QA) is flagged inline in the NCCN and GLKN sections as candidate appendix material, not removed.